# Qwen2-Audio INT8: xuất gói cho board KV260 (bước 2, v1 — Colab)

Chạy lại đúng quy trình đã chốt ở v12 (làm mượt offline + hiệu chuẩn tĩnh P1.5 trên 50 clip) rồi **xuất gói INT8** vào `<PKG_DIR>/`:
trọng số INT8 `[K,N]`, tham số requant theo kênh `[N,4]` (đúng `npu_qparam_t`), tham số FP, scale, tokenizer, **vector kiểm thử từng GEMM** (cùng định dạng `kv260_all`) và **vector đầu-cuối** (token + top-5 logit).

Không cần GPU lớn (model khoảng 1,2 tỉ tham số). Notebook tự dừng nếu một bước kiểm tra nội bộ không khớp:
* đọc lại **mọi** GEMM và tensor từ file nhị phân, so chính xác với mô hình trong bộ nhớ;
* mỗi `state_dict` đều được xuất hoặc báo rõ (không sót tham số);
* vector GEMM (lấy từ một lần forward INT8 thật, prefill và decode) phải **khớp đúng** đầu ra mà mô phỏng trả về, trừ các hàng fallback FP;
* mel cài bằng numpy theo mô tả trong `FORMAT.md` so với bộ xử lý của HF (chỉ cảnh báo).

Cell 0 gắn Google Drive. Nếu bạn đã chạy v12 trên Colab, notebook tự tìm `gen_wavs` và `scales_P15.json` của lần đó để clip giống hệt và so scale (cell 6). Cần Internet (tải model HF, và edge-tts nếu chưa có wav).
Lần đầu có thể cần Restart kernel sau khi cài `transformers` rồi Run All lại. Gói **chưa chứng minh bit-true với RTL**: bước đó chạy trên board bằng `vectors/`.

In [1]:
# ===== 0. Cau hinh (Google Colab) =====
PLATFORM = "colab"
N_CLIPS = 50            # PHAI = 50 de scale giong lan chay v12 (2 = chi chay thu)
MAX_NEW = 200
MODEL_ID = "tranquangchung/qwen2-audio-dialogue"
REV = "01061daa248750e56b43801a3c319fb4e6443704"
REPO_URL = "https://github.com/tranquangchung/Audio_Speech_Conversation"
EMBED_DTYPE = "fp16"    # bang embedding luu 'fp16' (nua dung luong) hoac 'fp32'; cell 6 do anh huong len top-1 va chot gia tri nay trong mo hinh
GOLDEN_CLIPS = 5        # so clip xuat vector dau-cuoi (token + top-5 logit)
CHECK_CLIPS = 20        # so clip dung de do anh huong cua lam tron embedding
VEC_ROWS = 48           # so hang A cua moi vector kiem thu GEMM (prefill)
import os
ROOT = "/content"
WORK = "<PROJECT_DIR>"
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Khong gan duoc Google Drive (", type(e).__name__, ") -> dung o cuc bo, se MAT khi ngat ket noi")
    WORK = ROOT + "/Qwen2_Audio_INT8"
if not os.path.isdir(WORK): print("Tao thu muc moi:", WORK)
os.makedirs(WORK, exist_ok=True)
PKG_DIR = WORK + "/pkg_v1"                     # goi xuat ra (copy ca thu muc nay sang board)
# tim lai wav edge-tts da sinh o v12 (de clip giong het lan hieu chuan truoc); khong thay thi se sinh lai o WORK/gen_wavs
_cands = [WORK + "/gen_wavs", WORK + "/qint8_report_v12/gen_wavs", "<GEN_WAVS_DIR>", WORK + "/qint8_report/gen_wavs"]
GEN_DIR = next((c for c in _cands if os.path.isdir(c) and any(f.startswith("extra_") for f in os.listdir(c))), WORK + "/gen_wavs")
os.makedirs(GEN_DIR, exist_ok=True)
_sc = [WORK + "/qint8_report_v12/scales_P15.json", "<SCALES_P15_JSON>", WORK + "/scales_P15.json"]
SAVED_SCALES = next((c for c in _sc if os.path.exists(c)), None)
print("WORK =", WORK, "\nGEN_DIR =", GEN_DIR, "\nscales v12 co san:", SAVED_SCALES)

Mounted at /content/drive
WORK = <PROJECT_DIR> 
GEN_DIR = <GEN_WAVS_DIR> 
scales v12 co san: <SCALES_P15_JSON>


In [2]:
import os, sys, json, time, glob, subprocess, importlib.metadata
need = {"transformers": "4.55.0", "accelerate": "1.10.1", "huggingface-hub": "0.34.4", "librosa": "0.11.0", "soundfile": "0.13.1"}
miss = []
for n, v in need.items():
    try:
        if importlib.metadata.version(n) != v: miss.append(f"{n}=={v}")
    except importlib.metadata.PackageNotFoundError:
        miss.append(f"{n}=={v}")
if miss:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *miss], capture_output=True, text=True)
    print("pip:", miss, "->", r.returncode, r.stderr[-300:])
    print(">>> Neu vua cai transformers moi: Restart kernel roi Run All lai (chi can 1 lan).")
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Can bat GPU (Settings -> Accelerator)"
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(torch.cuda.get_device_name(0), round(gb, 1), "GB")

pip: ['transformers==4.55.0', 'accelerate==1.10.1', 'huggingface-hub==0.34.4', 'soundfile==0.13.1'] -> 0 3.0,>=1.16.0, but you have huggingface-hub 0.34.4 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.34.4 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.34.4 which is incompatible.

>>> Neu vua cai transformers moi: Restart kernel roi Run All lai (chi can 1 lan).
torch 2.11.0+cu130 | transformers 4.55.0 | cuda True
Tesla T4 14.6 GB


In [3]:
# ===== 1. Clone repo (14 wav goc) + sinh them wav bang edge-tts cho du N_CLIPS (luu o GEN_DIR) =====
REPO = ROOT + "/Audio_Speech_Conversation"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
BASE_QUESTIONS = [
 "If you had to live in a video game universe for a year, which one would you choose?",
 "Would you rather be able to speak every language fluently or play every musical instrument perfectly?",
 "If animals could talk, which species do you think would be the rudest?",
 "If you found a suitcase with $1 million inside but no ID, would you keep it or turn it in?",
 "If you were a ghost, who would you haunt and how?",
 "Would you rather travel 100 years into the past or 100 years into the future?",
 "If you could only eat one cuisine (Italian, Japanese, Mexican, etc.) for the rest of your life, what would it be?",
 "If you were forced to participate in the Olympics, which sport would you have the best chance of not embarrassing yourself in?",
 "If you could cure one disease or solve one world problem, which would you pick?",
 "If you had to delete all apps from your phone except for three, which ones would you keep?",
 "What is the capital city of Vietnam?",
 "Vietnam is consistently one of the world's top two exporters of which caffeinated commodity?",
 "What is the ancient name of Hanoi, which translates to Ascending Dragon",
 "In Vietnamese culture, what are the specific flowers used to decorate homes during the Tet holiday (Lunar New Year) in the North versus the South?"]
EXTRA_QUESTIONS = [
 "If you could have dinner with any historical figure, who would you invite?",
 "What is one skill you have always wanted to learn but never had the time for?",
 "If you could instantly master any scientific subject, which one would you choose?",
 "Would you rather explore the deepest parts of the ocean or travel into deep space?",
 "If you could teleport anywhere in the world right now for one hour, where would you go?",
 "What is the most memorable book or movie you have ever experienced?",
 "If you had a time machine that only worked once, would you use it?",
 "Which season of the year do you enjoy the most and why?",
 "If you could design your ideal weekend, what activities would it include?",
 "Would you rather live in a bustling modern city or a quiet cabin in the mountains?",
 "If you could switch jobs with anyone for a single day, whose job would you try?",
 "What do you think will be the biggest technological breakthrough in the next ten years?",
 "If you had to teach a class on any topic, what subject would you teach?",
 "Would you rather have a rewind button or a pause button for your life?",
 "If you could bring back one extinct animal species, which one would it be?",
 "What is your favorite way to relax after a long and busy day?",
 "If you could only listen to three musical artists for the rest of your life, who would they be?",
 "Would you rather never be stuck in traffic again or never catch a cold again?",
 "If you could invent a new holiday, what would it celebrate?",
 "What is one piece of advice you would give to someone starting university?",
 "Which planet in our solar system is known as the Red Planet?",
 "What is the largest ocean on Earth?",
 "Who wrote the famous play Romeo and Juliet?",
 "What is the chemical symbol for gold?",
 "Which country is home to the Great Barrier Reef?",
 "How many continents are there on Earth?",
 "What is the fastest land animal in the world?",
 "Which traditional Vietnamese noodle soup is famous around the world?",
 "What is the name of the famous bay in northern Vietnam known for its emerald waters and limestone islands?",
 "In computer science, what does the acronym CPU stand for?",
 "Which gas do plants absorb from the atmosphere during photosynthesis?",
 "What is the tallest mountain in the world above sea level?",
 "Which instrument has eighty-eight keys?",
 "At what temperature in Celsius does water boil at standard atmospheric pressure?",
 "What is the currency used in Japan?",
 "Which organ in the human body is responsible for pumping blood?"]
WAV_DIR = REPO + "/LLM_Question"
ORIG = sorted(p for p in glob.glob(WAV_DIR + "/*.wav") if not os.path.basename(p).startswith("extra_"))
assert len(ORIG) == len(BASE_QUESTIONS), f"repo co {len(ORIG)} wav, can {len(BASE_QUESTIONS)}"
EXTRA_PATHS = [f"{GEN_DIR}/extra_{k:03d}.wav" for k in range(len(BASE_QUESTIONS) + 1, len(BASE_QUESTIONS) + len(EXTRA_QUESTIONS) + 1)]
n_extra = max(0, N_CLIPS - len(ORIG))
todo = [(p, q, len(BASE_QUESTIONS) + 1 + j) for j, (p, q) in enumerate(zip(EXTRA_PATHS[:n_extra], EXTRA_QUESTIONS[:n_extra])) if not os.path.exists(p)]

def tts_all(todo):
    # Ten file co dinh theo so thu tu cau (chay lai khong sinh trung); ghi file tam roi doi ten; thu lai 4 lan moi cau.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-U", "edge-tts"], check=True)
    import asyncio, threading, edge_tts, librosa, soundfile as sf
    voices = ["en-US-GuyNeural", "en-US-JennyNeural"]
    async def one(path, text, voice):
        tmp_mp3, tmp_wav = path + ".mp3", path + ".part.wav"
        await edge_tts.Communicate(text, voice).save(tmp_mp3)
        y, _ = librosa.load(tmp_mp3, sr=16000, mono=True)
        assert len(y) > 8000, "audio qua ngan"
        sf.write(tmp_wav, y, 16000, format="WAV")
        os.replace(tmp_wav, path); os.remove(tmp_mp3)
    async def main():
        for path, text, k in todo:
            last = None
            for attempt in range(4):
                try:
                    await one(path, text, voices[k % 2]); break
                except Exception as e:
                    last = e; await asyncio.sleep(2 * (attempt + 1))
            else:
                print("LOI tao", os.path.basename(path), type(last).__name__, str(last)[:120])
    err = {}
    def worker():                                        # luong rieng: tranh loi 'event loop is already running' cua Jupyter
        try: asyncio.run(main())
        except Exception as e: err["e"] = e
    th = threading.Thread(target=worker); th.start(); th.join()
    if err: print("LOI edge-tts:", type(err["e"]).__name__, str(err["e"])[:200])

if todo:
    print(f"Dang tao them {len(todo)} file wav (edge-tts, can Internet)...")
    t0 = time.time(); tts_all(todo); print(f"xong [{time.time()-t0:.0f}s]")
PAIRS = [(p, BASE_QUESTIONS[i]) for i, p in enumerate(ORIG)]
PAIRS += [(p, q) for p, q in zip(EXTRA_PATHS, EXTRA_QUESTIONS) if os.path.exists(p)]
PAIRS = PAIRS[:N_CLIPS]
WAVS, QUESTIONS = [p for p, _ in PAIRS], [q for _, q in PAIRS]
N_ORIG = min(len(ORIG), len(PAIRS))
print(len(WAVS), "file wav (", N_ORIG, "goc +", len(WAVS) - N_ORIG, "moi) / yeu cau", N_CLIPS)
if len(WAVS) < N_CLIPS: print("CANH BAO: thieu", N_CLIPS - len(WAVS), "file (edge-tts loi?). Tiep tuc voi so clip hien co.")
assert len(WAVS) >= 2

50 file wav ( 14 goc + 36 moi) / yeu cau 50


## 2. Module mô phỏng `qint8_sim.py`

In [4]:
%%writefile qint8_sim.py
"""
qint8_sim.py - Mo phong INT8 (PL + CPU) cho Qwen2-Audio tren GPU bang torch.

Muc dich: do CHAT LUONG (khop token so voi model FP) cua chuoi INT8 "tinh" truoc khi dua len board.
Day la mo phong, KHONG phai bit-true voi RTL: phep nhan tich luy so nguyen la chinh xac (INT8xINT8 -> INT32),
requant dung cung cong thuc (mult, shift) voi qffn_common.make_qparams / requant_golden, nhung buoc
nhan he so diem (QK) va softmax/AV dung float32 nen co the lech 1 LSB o vai gia tri hoa tie.

Cac tensor quan tam (site):
  <ten>.in   dau vao GEMM (luong tu hoa tinh, doi xung, per-tensor)
  <ten>.out  dau ra GEMM sau requant INT8 (per-tensor), tru lm_head khi lm_out == 'int32'
  <attn>.q, <attn>.k  (chi LLM, sau RoPE)   <attn>.score  (diem QK sau requant INT8, khi score_mode == 'int8')
Che do FALLBACK HANG NGOAI LAI (QS.row_fb): vai token (token dau chuoi `<|audio_bos|>`, vai `<|AUDIO|>`) co kich hoat
lon bat thuong (massive activations). Khi hieu chuan, cac hang do (max hang > row_c * trung vi max hang) bi loai khoi thong ke
scale; khi chay, hang nao se bao hoa (|x| > 127.5*scale vao, hoac dau ra cham 127/-128) duoc tinh bang FP tren CPU voi trong so
INT8 giai luong tu (W8*sw), khong qua PL. Cac vi tri bao hoa cung duoc tinh FP trong attention (cot K/V, hang Q).
Trong so: per-channel (moi kenh dau ra mot scale). Phan khong nhan ma tran (norm, RoPE, softmax, GELU/SiLU, cong
residual, embedding) chay float32 nhu tren CPU.
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
try:
    torch.set_float32_matmul_precision("highest")
except Exception:
    pass

PCTS = (100.0, 99.999, 99.99, 99.9, 99.5)
KCHUNK = 1024  # 1024*127*128 < 2^24  => tong tung khoi float32 la so nguyen chinh xac


class QState:
    def __init__(self):
        self.mode = "off"          # 'off' (FP goc) | 'calib' (FP + ghi thong ke) | 'quant' (INT8)
        self.scales = {}           # site -> scale kich hoat
        self.stats = {}            # site -> [tensor(len(PCTS))]  (thong ke cua clip hien tai)
        self.valid = {}            # 'mel','enc','proj' -> so hang hop le
        self.fp_layers = set()     # khoa lop giu FP, vd 'enc.5', 'llm.0', 'conv1', 'proj', 'lm_head'
        self.score_mode = "int8"   # 'int8' | 'int32'
        self.lm_out = "int32"      # 'int8' | 'int32'
        self.collect = None        # dict: ten lop -> tensor dau ra (cpu) de do loi tung lop
        self.row_fb = False        # fallback FP cho hang ngoai lai
        self.row_c = 10.0          # hieu chuan: hang ngoai lai neu max hang > row_c * trung vi(max hang)
        self.fb_w = "int8"         # trong so cho hang FP: 'int8' (W8*sw) | 'fp' (trong so goc, can tren)
        self.pos_flag = {}         # layer_key -> bool [B,T]: vi tri ngoai lai cua q/k/v_proj lop do
        self.fb = {}               # ten -> [so hang/vi tri FP, tong] (dem trong calib va quant)
        self.pf_cache = {}         # layer_key -> bool [B,S]: co vi tri ngoai lai cua toan bo khoa K/V (cho decode co KV cache)
        self.int32_ops = ()        # hau to ten GEMM tra ve INT32 (PS giai luong tu bang FP, khong requant INT8), '*' = tat ca
        self.out_pc = False        # dau ra INT8 dung scale THEO KENH dau ra (requant PL da co mult/shift theo kenh; PS giai luong tu bang vector)
        self.fb_phase = {}         # {'pre': [fp, tong], 'dec': [fp, tong]}: hang FP tach theo prefill (nhieu hang) va decode (1 hang/buoc), chi o che do quant, khong tinh attention
        self.sat = "edge"          # co bao hoa dau ra INT8: 'edge' = ket qua ==127/-128 (cu; voi scale theo kenh = max hieu chuan thi bao nhieu hang vo ly) | 'clip' = gia tri that vuot 127.5 (bi cat that)
        self.attn_fp = ()          # chan doan attention: tap con cua {'q','k','v','p'} giu FP (khong luong tu toan hang do)


QS = QState()


# ----------------------------------------------------------------------------- thong ke
@torch.no_grad()
def _stat(x):
    a = x.detach().abs().float().flatten()
    n = a.numel()
    if n == 0:
        return torch.zeros(len(PCTS))
    mx = a.max().view(1)
    step = max(1, n // 2_000_000)
    if step > 1 and step % 2 == 0:
        step += 1
    if step > 1:
        a = a[::step]
    qs = torch.tensor([p / 100.0 for p in PCTS[1:]], device=a.device, dtype=torch.float32)
    q = torch.quantile(a, qs)
    return torch.cat([mx, q]).cpu()


def record(site, x):
    QS.stats.setdefault(site, []).append(_stat(x))


def record_vec(site, y2):
    """Thong ke theo kenh dau ra: max |y| tung kenh cua cac hang y2 [R,N]."""
    QS.stats.setdefault(site, []).append(y2.detach().abs().amax(0).float().cpu())


def start_calib():
    QS.stats = {}
    QS.fb = {}
    QS.pos_flag = {}
    QS.pf_cache = {}


def reset_fb():
    QS.fb = {}
    QS.fb_phase = {}
    QS.pos_flag = {}
    QS.pf_cache = {}


def _note(name, nfb, n):
    a = QS.fb.setdefault(name, [0, 0])
    a[0] += int(nfb)
    a[1] += int(n)
    if QS.mode == "quant" and not name.endswith(".attn"):
        b = QS.fb_phase.setdefault("dec" if int(n) == 1 else "pre", [0, 0])
        b[0] += int(nfb)
        b[1] += int(n)


def fb_split():
    """Ti le hang FP (%) cua prefill va decode: {'pre': (%, tong hang), 'dec': (%, tong hang)}."""
    return {k: (100.0 * v[0] / max(v[1], 1), v[1]) for k, v in QS.fb_phase.items()}


def fb_report(top=12, prefix=None):
    """In ti le hang/vi tri duoc chay FP (QS.fb tich luy tu luc reset)."""
    rows = [(k, v[0], v[1]) for k, v in QS.fb.items() if v[1] > 0 and (prefix is None or k.startswith(prefix))]
    tot_f, tot_n = sum(r[1] for r in rows), sum(r[2] for r in rows)
    print(f"  FP-hang tong: {tot_f}/{tot_n} = {100.0 * tot_f / max(tot_n, 1):.2f}%")
    for k, f, n in sorted(rows, key=lambda r: -r[1] / r[2])[:top]:
        print(f"    {k:22s} {f:6d}/{n:<7d} = {100.0 * f / n:6.2f}%")
    return rows


def _row_keep(x2):
    """Hieu chuan: hang binh thuong = max hang <= row_c * trung vi(max hang)."""
    r = x2.detach().abs().amax(1).float()
    med = r.median()
    if float(med) <= 0:
        return torch.ones_like(r, dtype=torch.bool)
    return r <= QS.row_c * med


def stats_of_clip():
    """Gop thong ke cua 1 clip: site -> tensor[len(PCTS)] (lay max neu site goi nhieu lan)."""
    return {k: torch.stack(v).max(0).values for k, v in QS.stats.items()}


def group_of(site):
    return "llm" if (site.startswith("llm.") or site.startswith("lm_head")) else "enc"


def build_scales(clip_stats, pct_enc=100.0, pct_llm=100.0, pct_map=None, agg="max", pc_margin=1.0):
    """clip_stats: list cac dict (site -> tensor). pct_map: list (chuoi con, pct) uu tien truoc nhom mac dinh."""
    sites = set()
    for d in clip_stats:
        sites |= set(d.keys())
    out = {}
    for s in sorted(sites):
        if s.endswith(".outc"):                          # scale theo kenh dau ra: max qua cac clip / 127
            mx = torch.stack([d[s] for d in clip_stats if s in d]).max(0).values
            out[s] = torch.where(mx > 0, mx * pc_margin / 127.0, torch.ones_like(mx))     # pc_margin > 1: du dia cho kenh co max thuc te nhinh hon hieu chuan
            continue
        pct = pct_enc if group_of(s) == "enc" else pct_llm
        for sub, p in (pct_map or []):
            if sub in s:
                pct = p
                break
        pi = PCTS.index(pct)
        vals = torch.stack([d[s][pi] for d in clip_stats if s in d])
        amax = float(vals.max() if agg == "max" else vals.mean())
        out[s] = amax / 127.0 if amax > 0 else 1.0
    return out


# ----------------------------------------------------------------------------- so hoc so nguyen
def int_mm(x8, W8):
    """x8 [M,K] float32 (so nguyen), W8 [N,K] int8 -> [M,N] float64 (so nguyen chinh xac)."""
    M, K = x8.shape
    acc = None
    for k0 in range(0, K, KCHUNK):
        part = x8[:, k0:k0 + KCHUNK] @ W8[:, k0:k0 + KCHUNK].float().t()
        part = part.double()
        acc = part if acc is None else acc + part
    return acc


def qparams(real):
    """Vector hoa qffn_common.make_qparams: real [N] float64 -> (mult int64, shift int64)."""
    real = real.double()
    shift = torch.clamp(torch.floor(torch.log2((2 ** 31 - 1) / real)), max=62.0)
    mult = torch.round(real * torch.pow(2.0, shift))
    for _ in range(3):
        bad = (mult > 2 ** 31 - 1) & (shift > 0)
        shift = torch.where(bad, shift - 1, shift)
        mult = torch.round(real * torch.pow(2.0, shift))
    assert bool((shift >= 0).all()) and bool((mult >= 1).all()), "he so requant ngoai pham vi"
    return mult.long(), shift.long()


def requant(acc, mult, shift, bias_i=None, zp=0):
    """acc float64 [M,N] (so nguyen). Tra ve int64 da clamp [-128,127]: ((acc+bias)*mult + rnd) >> shift + zp."""
    t = acc.long()
    if bias_i is not None:
        t = t + bias_i
    rnd = torch.where(shift > 0, torch.bitwise_left_shift(torch.ones_like(shift), (shift - 1).clamp(min=0)),
                      torch.zeros_like(shift))
    v = torch.bitwise_right_shift(t * mult + rnd, shift) + zp
    return torch.clamp(v, -128, 127)


# ----------------------------------------------------------------------------- GEMM co luong tu
class QGemm(nn.Module):
    """Co so cho QLinear / QConv1d: trong so INT8 per-channel + quy trinh luong tu."""

    ROWFB = True        # QConv1d tat

    def __init__(self, name, layer_key, weight_nk, bias):
        super().__init__()
        self.name = name
        self.layer_key = layer_key
        self.valid_key = None
        self.last_fb = None
        W = weight_nk.detach()
        N, K = W.shape
        W8 = torch.empty((N, K), dtype=torch.int8, device=W.device)
        sw = torch.empty(N, dtype=torch.float64, device=W.device)
        for n0 in range(0, N, 8192):
            w = W[n0:n0 + 8192].double()
            amax = w.abs().amax(dim=1)
            s = torch.where(amax > 0, amax / 127.0, torch.ones_like(amax))
            W8[n0:n0 + 8192] = torch.clamp(torch.round(w / s[:, None]), -128, 127).to(torch.int8)
            sw[n0:n0 + 8192] = s
        self.register_buffer("W8", W8, persistent=False)
        self.register_buffer("sw", sw, persistent=False)
        self.bias = None if bias is None else bias.detach().double()
        self._cache = None

    @property
    def fp(self):
        return self.layer_key in QS.fp_layers or self.name in QS.fp_layers

    @property
    def s_out(self):
        return QS.scales[self.name + ".out"]

    def sout_vec(self):
        """Scale dau ra theo kenh (tensor float64 tren thiet bi) neu QS.out_pc, nguoc lai None."""
        if not QS.out_pc:
            return None
        src = QS.scales[self.name + ".outc"]
        if getattr(self, "_ov", None) is None or self._ov[0] is not src:
            self._ov = (src, src.to(self.W8.device).double())
        return self._ov[1]

    def vscale(self):
        """Scale dau ra dung cho V trong attention: vector [N] float32 (out_pc) hoac so vo huong."""
        v = self.sout_vec()
        return self.s_out if v is None else v.float()

    def _qp(self, sin, sout):
        key = (sin, id(sout) if torch.is_tensor(sout) else sout)
        if self._cache is None or self._cache[0] != key:
            mult, shift = qparams(sin * self.sw / sout)
            bi = None
            if self.bias is not None:
                bi = torch.clamp(torch.round(self.bias / (sin * self.sw)), -(2 ** 31), 2 ** 31 - 1).long()
            self._cache = (key, mult, shift, bi)
        return self._cache[1:]

    def _wfp(self):
        raise NotImplementedError

    def _fp_rows(self, xf):
        """Hang ngoai lai tren CPU: x FP32 @ W (INT8 giai luong tu, hoac FP goc neu QS.fb_w == 'fp') + bias."""
        N = self.W8.shape[0]
        if QS.fb_w == "fp":
            y = xf @ self._wfp().float().t()
        else:
            y = torch.empty((xf.shape[0], N), dtype=torch.float32, device=xf.device)
            for n0 in range(0, N, 8192):
                Wd = self.W8[n0:n0 + 8192].float() * self.sw[n0:n0 + 8192].float()[:, None]
                y[:, n0:n0 + 8192] = xf @ Wd.t()
        if self.bias is not None:
            y = y + self.bias.float()
        return y

    def gemm(self, x2d, int32_out=False):
        """x2d [M,K] float -> [M,N] float32 (da dequant)."""
        sin = QS.scales[self.name + ".in"]
        xf = x2d.float()
        use_fb = QS.row_fb and self.ROWFB
        x8 = torch.clamp(torch.round(xf / sin), -128, 127)
        acc = int_mm(x8, self.W8)
        fb = (xf.abs().amax(1) > 127.5 * sin) if use_fb else None
        if int32_out:
            if self.bias is not None:
                acc = acc + torch.clamp(torch.round(self.bias / (sin * self.sw)), -(2 ** 31), 2 ** 31 - 1)
            y = (acc * (sin * self.sw)).float()
        else:
            sout = self.sout_vec()
            if sout is None:
                sout = QS.scales[self.name + ".out"]
            mult, shift, bi = self._qp(sin, sout)
            v = requant(acc, mult, shift, bi)
            if use_fb:
                if QS.sat == "clip":
                    raw = (acc + (bi if bi is not None else 0)) * (sin * self.sw / sout)
                    fb = fb | (raw.abs() > 127.5).any(dim=1)
                else:
                    fb = fb | ((v == 127) | (v == -128)).any(dim=1)
            y = v.float() * (sout.float() if torch.is_tensor(sout) else sout)
        if use_fb:
            self.last_fb = fb
            n = QS.valid.get(self.valid_key) if self.valid_key else None
            nv = fb.numel() if n is None else min(n, fb.numel())
            _note(self.name, fb[:nv].sum(), nv)
            if bool(fb.any()):
                y[fb] = self._fp_rows(xf[fb])
        return y


def _rows(x, key):
    n = QS.valid.get(key) if key else None
    return x if n is None else x[:, :n]


class QLinear(QGemm):
    def __init__(self, name, layer_key, lin, valid_key=None):
        super().__init__(name, layer_key, lin.weight, lin.bias)
        self.lin = lin
        self.valid_key = valid_key
        self._isq = name.endswith(".q_proj")
        self._isqkv = name.endswith((".q_proj", ".k_proj", ".v_proj"))

    @property
    def weight(self):                 # HF doc .weight o mot so cho
        return self.lin.weight

    def _wfp(self):
        return self.lin.weight

    def _set_pf(self, fbfull):
        """fbfull bool [B,T] hoac None. q_proj dat lai, k/v_proj hop vao (de attention biet vi tri ngoai lai)."""
        if not self._isqkv:
            return
        cur = None if self._isq else QS.pos_flag.get(self.layer_key)
        if fbfull is not None:
            cur = fbfull if cur is None else (cur | fbfull)
        QS.pos_flag[self.layer_key] = cur

    def forward(self, x):
        if self._isq:
            QS.pos_flag[self.layer_key] = None
        if QS.mode == "off" or self.fp:
            return self.lin(x)
        if QS.mode == "calib":
            if QS.row_fb and self.ROWFB:
                xs = _rows(x, self.valid_key)                    # [B,n,K]
                B_, n_, K_ = xs.shape
                x2 = xs.reshape(-1, K_)
                keep = _row_keep(x2)
                y = self.lin(x)
                ys = _rows(y, self.valid_key).reshape(-1, y.shape[-1])
                record(self.name + ".in", x2[keep])
                record(self.name + ".out", ys[keep])
                if QS.out_pc:
                    record_vec(self.name + ".outc", ys[keep])
                _note(self.name, (~keep).sum(), keep.numel())
                if self._isqkv:
                    pf = torch.zeros(x.shape[0], x.shape[1], dtype=torch.bool, device=x.device)
                    pf[:, :n_] = (~keep).view(B_, n_)
                    self._set_pf(pf)
                return y
            record(self.name + ".in", _rows(x, self.valid_key))
            y = self.lin(x)
            record(self.name + ".out", _rows(y, self.valid_key))
            if QS.out_pc:
                ym = _rows(y, self.valid_key)
                record_vec(self.name + ".outc", ym.reshape(-1, ym.shape[-1]))
            return y
        shp = x.shape
        int32 = (self.name == "lm_head" and QS.lm_out == "int32") or any(
            o == "*" or self.name.endswith(o) for o in QS.int32_ops)
        y = self.gemm(x.reshape(-1, shp[-1]), int32_out=int32)
        if QS.row_fb and self._isqkv and self.last_fb is not None:
            self._set_pf(self.last_fb.view(shp[0], shp[1]))
        return y.reshape(*shp[:-1], -1)


class QConv1d(QGemm):
    ROWFB = False

    def __init__(self, name, layer_key, conv, key_in, key_out):
        w = conv.weight.detach()                      # [out, C, ks]
        super().__init__(name, layer_key, w.permute(0, 2, 1).reshape(w.shape[0], -1), conv.bias)
        self.conv = conv
        self.ks = conv.kernel_size[0]
        self.stride = conv.stride           # tuple: encoder doc self.conv1.stride[0]
        self.pad = conv.padding[0]
        self.key_in, self.key_out = key_in, key_out

    @property
    def weight(self):
        return self.conv.weight

    def _wfp(self):
        w = self.conv.weight
        return w.permute(0, 2, 1).reshape(w.shape[0], -1)

    def _im2col(self, x):                              # x [B,C,T] -> [B,Tout,ks*C], chi so k*C + c
        xp = F.pad(x, (self.pad, self.pad))
        st = self.stride[0]
        tout = (xp.shape[-1] - self.ks) // st + 1
        cols = [xp[:, :, k:k + st * (tout - 1) + 1:st].transpose(1, 2) for k in range(self.ks)]
        return torch.cat(cols, dim=-1)

    def forward(self, x):
        if QS.mode == "off" or self.fp:
            return self.conv(x)
        if QS.mode == "calib":
            n = QS.valid.get(self.key_in)
            record(self.name + ".in", x[:, :, :n] if n else x)
            y = self.conv(x)
            m = QS.valid.get(self.key_out)
            record(self.name + ".out", y[:, :, :m] if m else y)
            if QS.out_pc:
                yc = y[:, :, :m] if m else y
                record_vec(self.name + ".outc", yc.transpose(1, 2).reshape(-1, yc.shape[1]))
            return y
        A = self._im2col(x)
        B_, T_, K_ = A.shape
        y = self.gemm(A.reshape(-1, K_)).reshape(B_, T_, -1)
        return y.transpose(1, 2).contiguous()


# ----------------------------------------------------------------------------- attention
def _chunked_av(P8, V8):
    """P8 [B,H,T,S] , V8 [B,H,S,d] (so nguyen, float32) -> float64 [B,H,T,d] chinh xac."""
    S_ = P8.shape[-1]
    acc = None
    for s0 in range(0, S_, KCHUNK):
        part = (P8[..., s0:s0 + KCHUNK] @ V8[..., s0:s0 + KCHUNK, :]).double()
        acc = part if acc is None else acc + part
    return acc


def _score_sel(S_, mask, kind, pf=None):
    """Chon phan tu hop le de lay thong ke diem (bo cac hang/cot o vi tri ngoai lai pf [B,T])."""
    if kind == "enc":
        n = QS.valid.get("enc")
        S_ = S_[:, :, :n, :n]
        if pf is not None and bool(pf[:, :n].any()):
            keep = ~pf[0, :n]
            S_ = S_[:, :, keep][:, :, :, keep]
        return S_
    if mask is None:
        ok = torch.ones_like(S_, dtype=torch.bool)
    else:
        ok = (mask[:, :, :, :S_.shape[-1]] > -1e30).expand_as(S_)
    if pf is not None and bool(pf.any()):
        ok = ok & ~(pf[:, None, :, None] | pf[:, None, None, :])
    return S_[ok]


def make_attention(orig, kind):
    def fn(module, query, key, value, attention_mask, *args, **kwargs):
        lk = module.layer_key
        if QS.mode == "off" or lk in QS.fp_layers or module.qname in QS.fp_layers:
            return orig(module, query, key, value, attention_mask, *args, **kwargs)
        scaling = kwargs.get("scaling", None)
        if kind == "enc":
            scaling = 1.0                                  # q da nhan module.scaling truoc khi goi
        elif scaling is None:
            scaling = query.size(-1) ** -0.5
        rep = query.shape[1] // key.shape[1]
        mask = attention_mask
        S_len = key.shape[-2]
        mslice = None if mask is None else mask[:, :, :, :S_len]

        fbm = QS.row_fb
        pf = QS.pos_flag.get(lk) if fbm else None            # [B,T]: vi tri cua cac hang query hien tai
        pfk = None                                           # [B,S]: vi tri cua toan bo khoa/gia tri (ke ca KV cache)
        if pf is not None and pf.shape[1] != query.shape[2]:
            pf = None
        if pf is not None:
            if pf.shape[1] == key.shape[2]:                  # prefill / teacher-forced
                pfk = pf
            else:                                            # decode: noi them vao co cua cac vi tri truoc
                prev = QS.pf_cache.get(lk)
                pfk = torch.cat([prev, pf], 1) if (prev is not None and prev.shape[1] + pf.shape[1] == key.shape[2]) else None
            QS.pf_cache[lk] = pfk
        if QS.mode == "calib":
            k = key.repeat_interleave(rep, 1) if rep > 1 else key
            v = value.repeat_interleave(rep, 1) if rep > 1 else value
            if kind == "llm":
                if pf is not None and bool(pf.any()):
                    kp = ~pf[0]
                    record(module.qname + ".q", query[:, :, kp])
                    record(module.qname + ".k", key[:, :, kp])
                else:
                    record(module.qname + ".q", query)
                    record(module.qname + ".k", key)
            sc = torch.matmul(query, k.transpose(2, 3)) * scaling
            record(module.qname + ".score", _score_sel(sc, mask, kind, pf))
            if pf is not None:
                nv = QS.valid.get("enc") if kind == "enc" else pf.shape[1]
                _note(module.qname, pf[0, :nv].sum(), nv)
            if mslice is not None:
                sc = sc + mslice
            p = F.softmax(sc, dim=-1, dtype=torch.float32).to(query.dtype)
            return torch.matmul(p, v).transpose(1, 2).contiguous(), None

        # ---- quant
        v_scale = module.v_proj.vscale()
        if kind == "enc":
            sq = module.q_proj.s_out * module.scaling
            sk = module.k_proj.s_out
            fac = sq * sk                                  # scaling da nam trong sq
        else:
            sq = QS.scales[module.qname + ".q"]
            sk = QS.scales[module.qname + ".k"]
            fac = sq * sk * scaling
        k = key.repeat_interleave(rep, 1) if rep > 1 else key
        v = value.repeat_interleave(rep, 1) if rep > 1 else value
        if torch.is_tensor(v_scale):                       # scale V theo kenh: [1,H,1,d]
            v_scale = v_scale.view(1, value.shape[1], 1, value.shape[3])
            v_scale = v_scale.repeat_interleave(rep, 1) if rep > 1 else v_scale
        af = QS.attn_fp
        q8 = query.float() / sq if "q" in af else torch.clamp(torch.round(query.float() / sq), -128, 127)
        k8 = k.float() / sk if "k" in af else torch.clamp(torch.round(k.float() / sk), -128, 127)
        v8 = v.float() / v_scale if "v" in af else torch.clamp(torch.round(v.float() / v_scale), -128, 127)
        acc = torch.matmul(q8, k8.transpose(2, 3))         # tong <= 64*128*128 < 2^24: chinh xac
        if QS.score_mode == "int8":
            ssc = QS.scales[module.qname + ".score"]
            sc = torch.clamp(torch.round(acc * (fac / ssc)), -128, 127) * ssc
        else:
            sc = acc * fac
        kcol = None
        if fbm:
            # vi tri bao hoa: Q (hang), K/V (cot) -> tinh FP tren CPU
            qrow = query.float().abs().amax(-1) > 127.5 * sq                       # [B,H,T]
            kcol = (k.float().abs().amax(-1) > 127.5 * sk) | ((v.float().abs() > 127.5 * v_scale).any(-1))   # [B,H,S]
            if pf is not None:
                qrow = qrow | pf[:, None, :]
            if pfk is not None:
                kcol = kcol | pfk[:, None, :]
            nv = QS.valid.get("enc") if kind == "enc" else qrow.shape[-1]
            _note(module.qname, (qrow.any(1) | kcol.any(1))[0, :nv].sum(), nv)
            if bool(qrow.any()) or bool(kcol.any()):
                fpm = qrow[..., :, None] | kcol[..., None, :]
                sc_fp = torch.matmul(query.float(), k.float().transpose(2, 3)) * scaling
                sc = torch.where(fpm, sc_fp, sc)
            else:
                kcol = None
        if mslice is not None:
            sc = sc + mslice
        p = F.softmax(sc, dim=-1, dtype=torch.float32)
        pn = p if kcol is None else p.masked_fill(kcol[..., None, :], 0.0)   # cot FP khong tham gia scale cua P
        sp = (pn.amax(dim=-1, keepdim=True) / 127.0).clamp_min(1e-30)
        p8 = pn / sp if "p" in af else torch.round(pn / sp)
        o = _chunked_av(p8, v8) * sp.double() * (v_scale.double() if torch.is_tensor(v_scale) else v_scale)
        if kcol is not None:
            pfp = p.masked_fill(~kcol[..., None, :], 0.0)
            o = o + torch.matmul(pfp, v.float()).double()
        return o.float().transpose(1, 2).contiguous(), None
    return fn


# ----------------------------------------------------------------------------- cai dat vao model
_ORIG = {}


def install(model):
    """Boc cac GEMM cua model bang QLinear/QConv1d va thay ham attention. Goi 1 lan; model o che do FP giu nguyen ket qua."""
    import transformers.models.qwen2_audio.modeling_qwen2_audio as qa
    import transformers.models.qwen2.modeling_qwen2 as q2
    at = model.audio_tower
    lm = model.language_model
    assert at.config._attn_implementation == "eager" and lm.config._attn_implementation == "eager", \
        "can attn_implementation='eager' cho ca audio_tower va language_model"
    if "qa" not in _ORIG:
        _ORIG["qa"], _ORIG["q2"] = qa.eager_attention_forward, q2.eager_attention_forward
    qa.eager_attention_forward = make_attention(_ORIG["qa"], "enc")
    q2.eager_attention_forward = make_attention(_ORIG["q2"], "llm")

    at.conv1 = QConv1d("conv1", "conv1", at.conv1, "mel", "mel")
    at.conv2 = QConv1d("conv2", "conv2", at.conv2, "mel", "enc")
    for i, L in enumerate(at.layers):
        lk = f"enc.{i}"
        a = L.self_attn
        for nm in ("q_proj", "k_proj", "v_proj", "out_proj"):
            setattr(a, nm, QLinear(f"{lk}.{nm}", lk, getattr(a, nm), "enc"))
        L.fc1 = QLinear(f"{lk}.fc1", lk, L.fc1, "enc")
        L.fc2 = QLinear(f"{lk}.fc2", lk, L.fc2, "enc")
        a.qname, a.layer_key = f"{lk}.attn", lk
        L.qname = lk
        L.register_forward_hook(_collect_hook("enc"))
    lin = model.multi_modal_projector.linear
    model.multi_modal_projector.linear = QLinear("proj", "proj", lin, "proj")
    for j, L in enumerate(lm.model.layers):
        lk = f"llm.{j}"
        a, m = L.self_attn, L.mlp
        for nm in ("q_proj", "k_proj", "v_proj", "o_proj"):
            setattr(a, nm, QLinear(f"{lk}.{nm}", lk, getattr(a, nm)))
        for nm in ("gate_proj", "up_proj", "down_proj"):
            setattr(m, nm, QLinear(f"{lk}.{nm}", lk, getattr(m, nm)))
        a.qname, a.layer_key = f"{lk}.attn", lk
        L.qname = lk
        L.register_forward_hook(_collect_hook("llm"))
    lm.lm_head = QLinear("lm_head", "lm_head", lm.lm_head)
    return model


def _collect_hook(kind):
    def hook(mod, inp, out):
        if QS.collect is None:
            return
        h = out[0] if isinstance(out, tuple) else out
        if kind == "enc":
            n = QS.valid.get("enc")
            h = h[:, :n]
        QS.collect[mod.qname] = h.detach().float().cpu()
    return hook


def set_valid(feature_attention_mask):
    L = int(feature_attention_mask.sum(-1)[0])
    L2 = (L - 1) // 2 + 1
    Lp = (L2 - 2) // 2 + 1
    QS.valid = {"mel": L, "enc": L2, "proj": Lp}
    return QS.valid


# ----------------------------------------------------------------------------- tien ich do chat luong
def apply_penalty(logits_row, history_ids, penalty):
    idx = torch.unique(history_ids)
    v = logits_row[idx]
    logits_row[idx] = torch.where(v > 0, v / penalty, v * penalty)
    return logits_row


@torch.no_grad()
def forward_logits(model, inputs, gen_ids):
    """Teacher-forced: tra ve (logits [n_gen,V] cho cac vi tri du doan gen_ids, input_len)."""
    ids = inputs["input_ids"]
    n0 = ids.shape[1]
    full = torch.cat([ids, gen_ids.view(1, -1).to(ids.device)], dim=1)
    out = model(input_ids=full, attention_mask=torch.ones_like(full),
                input_features=inputs["input_features"], feature_attention_mask=inputs["feature_attention_mask"],
                use_cache=False)
    lg = out.logits[0, n0 - 1: full.shape[1] - 1].float()
    return lg, n0, full[0]


def compare_logits(lf, lq, full_ids, n0, penalty=1.1):
    """lf, lq [n,V] (logit tho). Ap dung cung phat lap lai cho 2 ben (lich su = prompt + token da sinh)."""
    n = lf.shape[0]
    lf = lf.to(lq.device)
    agree = []
    rank5 = []
    for t in range(n):
        hist = full_ids[:n0 + t]
        a = apply_penalty(lf[t].clone(), hist, penalty)
        b = apply_penalty(lq[t].clone(), hist, penalty)
        agree.append(bool(a.argmax() == b.argmax()))
        rank5.append(bool((b.topk(5).indices == a.argmax()).any()))
    agree = torch.tensor(agree)
    cos = F.cosine_similarity(lf, lq, dim=-1).mean().item()
    pf, lpq = F.softmax(lf, -1), F.log_softmax(lq, -1)
    kl = (pf * (torch.log(pf.clamp_min(1e-30)) - lpq)).sum(-1).mean().item()
    first = int((~agree).nonzero()[0]) if (~agree).any() else n
    return {"n": n, "top1": agree.float().mean().item(), "top5": torch.tensor(rank5).float().mean().item(),
            "cos": cos, "kl": kl, "first_div": first}

Writing qint8_sim.py


In [5]:
# ===== 3. Nạp model FP32 (eager attention) + processor =====
import gc
for _n in ("model", "REF", "STATS", "STATS_C", "STATS_NOSMOOTH", "CH", "lf0", "lf1"):
    globals().pop(_n, None)                                  # giai phong GPU neu chay lai o nay trong kernel cu
gc.collect(); torch.cuda.empty_cache()
print("GPU dang giu truoc khi nap: %.2f GB (phai gan 0)" % (torch.cuda.memory_allocated() / 2**30))
import numpy as np, librosa
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=REV)
model = Qwen2AudioForConditionalGeneration.from_pretrained(MODEL_ID, revision=REV, torch_dtype=torch.float32,
                                                           attn_implementation="eager").to("cuda").eval()
model.audio_tower.config._attn_implementation = "eager"
model.language_model.config._attn_implementation = "eager"
print("so tham so: %.0f M" % (sum(p.numel() for p in model.parameters()) / 1e6))
print("audio", model.config.audio_config.encoder_layers, "lop | LLM", model.config.text_config.num_hidden_layers, "lop, hidden",
      model.config.text_config.hidden_size, "| tie:", model.config.text_config.tie_word_embeddings)
IM_END_ID = processor.tokenizer.convert_tokens_to_ids("<|im_end|>")
AUDIO_ID = getattr(model.config, "audio_token_id", None) or model.config.audio_token_index
SR = processor.feature_extractor.sampling_rate
INSTR = "You are a virtual assistant, answering questions and providing information in a clear and polite manner."
PROMPT = ("<|audio_bos|><|AUDIO|><|audio_eos|><|im_start|>user\nAudio content provided. " + INSTR + "<|im_end|>\n<|im_start|>assistant\n")

def make_inputs(path):
    audio, _ = librosa.load(path, sr=SR)
    r = processor(text=PROMPT, audio=audio, return_tensors="pt", sampling_rate=SR, padding=True)
    return {k: v.to("cuda") for k, v in r.items()}

GPU dang giu truoc khi nap: 0.00 GB (phai gan 0)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/153 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/746 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

... (9 outputs left out)

model-00002-of-00002.safetensors:   0%|          | 0.00/559M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

so tham so: 1276 M
audio 32 lop | LLM 24 lop, hidden 896 | tie: False


In [6]:
# ===== 4. Tham chiếu FP: sinh greedy giống test_dialogue_inthewild.py =====
REF = []
for i, p in enumerate(WAVS):
    inp = make_inputs(p)
    t0 = time.time()
    with torch.no_grad():
        g = model.generate(**inp, max_new_tokens=MAX_NEW, do_sample=False, repetition_penalty=1.1, eos_token_id=IM_END_ID,
                           pad_token_id=processor.tokenizer.pad_token_id, use_cache=True)
    gen = g[0, inp["input_ids"].shape[1]:]
    text = processor.batch_decode(gen[None], skip_special_tokens=True)[0].strip()
    REF.append(dict(path=p, inp=inp, gen=gen, text=text))
    print(f"[{i+1:2d}] audio_tokens={int((inp['input_ids']==AUDIO_ID).sum())} prompt={inp['input_ids'].shape[1]} "
          f"sinh={len(gen)} ({time.time()-t0:.1f}s)\n     Q: {QUESTIONS[i] if i < len(QUESTIONS) else ''}\n     A: {text}")

[ 1] audio_tokens=172 prompt=204 sinh=91 (6.3s)
     Q: If you had to live in a video game universe for a year, which one would you choose?
     A: That's an interesting question! Personally, I think the universe would be incredibly peaceful if it were a good old-fashioned video game world. There wouldn't be any conflict or violence between players. On the other hand, some people might find that a more realistic or fair scenario could have more consequences. It really depends on what kind of balance you're looking for in such a setting. What about you? Which universe do you imagine being the most balanced?
[ 2] audio_tokens=186 prompt=218 sinh=90 (4.4s)
     Q: Would you rather be able to speak every language fluently or play every musical instrument perfectly?
     A: That's an interesting question! Personally, I think it would be fun to have the ability to speak every language fluently, as that could make travel even more exciting. On the other hand, playing all musical instruments p

In [7]:
# ===== 5. Boc model bang QLinear/QConv1d; kiem tra che do 'off' KHONG doi ket qua =====
import qint8_sim as sim
lf0, n0, full0 = sim.forward_logits(model, REF[0]["inp"], REF[0]["gen"])      # truoc khi boc
sim.install(model)
sim.QS.mode = "off"
lf1, _, _ = sim.forward_logits(model, REF[0]["inp"], REF[0]["gen"])
print("sau khi boc, che do off: max|diff| logit =", float((lf0 - lf1).abs().max()), "(phai = 0)")
assert float((lf0 - lf1).abs().max()) == 0.0
print("so GEMM duoc boc:", sum(isinstance(m, sim.QGemm) for m in model.modules()))

sau khi boc, che do off: max|diff| logit = 0.0 (phai = 0)
so GEMM duoc boc: 364


In [8]:
# ===== 5c. Logit FP32 teacher-forced (tham chieu cho vector dau-cuoi) =====
sim.QS.mode = "off"
for r in REF:
    lf, n0, full = sim.forward_logits(model, r["inp"], r["gen"]); r["lf"], r["n0"], r["full"] = lf.cpu(), n0, full
print("FP32:", len(REF), "clip,", sum(r["lf"].shape[0] for r in REF), "vi tri token")

FP32: 50 clip, 3410 vi tri token


## 2b. Module làm mượt `qsmooth.py`

In [9]:
%%writefile qsmooth.py
"""
qsmooth.py - Lam muot kenh (SmoothQuant) NGOAI TUYEN, tuong duong FP, cho bo mo phong qint8_sim.

Y tuong: mot vai kenh dau vao co bien do lon bat thuong (vd. dau vao down_proj) khien scale per-tensor qua lon. Chia kenh j cua
dau vao cho s_j va nhan cot j cua trong so voi s_j thi ket qua FP khong doi. Phan chia duoc GOP vao tham so lop truoc (khong them
phep tinh nao tren board):
  down_proj : PS nhan dau vao (silu(gate)*up) voi 1/s theo kenh (gop vao buoc nhan silu*up, them 1 phep nhan/phan tu), nhan cot
              down_proj voi s. KHONG gop vao up_proj: dau ra INT8 per-tensor cua up_proj se bi lech phan bo kenh (da do: top-1 cua up_proj
              tu 94% xuong 82%).
  gate, up  : chia trong so RMSNorm post_attention_layernorm cho s
  q, k, v   : chia trong so RMSNorm input_layernorm cho s
  q/k       : nhan hang q_proj voi t, chia hang k_proj cho t (cap kenh RoPE (i, i+d/2) dung chung t; GQA: t theo nhom kv)
  encoder   : LayerNorm cho qkv va fc1; v_proj (hang) cho out_proj
Moi thay doi deu hoan nguyen duoc bang restore().
"""
import torch

_ORIG = {}
_TOUCH = {}
_HOOKS = []
INV = {}          # 'llm.<j>.down_proj' -> vector 1/s (float32, cpu) ma PS nhan vao dau vao down_proj (de xuat goi INT8)


def _save(t):
    if id(t) not in _ORIG:
        _ORIG[id(t)] = (t, t.detach().cpu().clone())


def _touch(m):
    _TOUCH[id(m)] = m


@torch.no_grad()
def _refresh(m):
    """Luong tu lai trong so INT8 per-channel cua mot QLinear sau khi trong so FP bi doi."""
    W = m.lin.weight.detach()
    N = W.shape[0]
    for n0 in range(0, N, 8192):
        w = W[n0:n0 + 8192].double()
        amax = w.abs().amax(dim=1)
        s = torch.where(amax > 0, amax / 127.0, torch.ones_like(amax))
        m.W8[n0:n0 + 8192] = torch.clamp(torch.round(w / s[:, None]), -128, 127).to(torch.int8)
        m.sw[n0:n0 + 8192] = s
    m.bias = None if m.lin.bias is None else m.lin.bias.detach().double()
    m._cache = None


def _mul_cols(m, s):                 # W[:, j] *= s[j]
    _save(m.lin.weight); _touch(m)
    m.lin.weight.data *= s.to(m.lin.weight.device, m.lin.weight.dtype)[None, :]


def _scale_rows(m, s, inverse):      # W[i, :] (va bias[i]) *= s[i]  hoac /= s[i]
    _save(m.lin.weight); _touch(m)
    sd = s.to(m.lin.weight.device, m.lin.weight.dtype)
    f = (1.0 / sd) if inverse else sd
    m.lin.weight.data *= f[:, None]
    if m.lin.bias is not None:
        _save(m.lin.bias)
        m.lin.bias.data *= f


def _div_vec(t, s):
    _save(t)
    t.data /= s.to(t.device, t.dtype)


def _smooth(X, Wcol, a):
    X, Wcol = X.to(Wcol.device).float().clamp_min(1e-5), Wcol.float().clamp_min(1e-5)
    s = X.pow(a) / Wcol.pow(1.0 - a)
    return s.clamp(1e-3, 1e3)


@torch.no_grad()
def restore(model):
    for h in _HOOKS:
        h.remove()
    _HOOKS.clear()
    INV.clear()
    for t, c in _ORIG.values():
        t.data.copy_(c.to(t.device))
    _ORIG.clear()
    for m in list(_TOUCH.values()):
        _refresh(m)
    _TOUCH.clear()


@torch.no_grad()
def collect_chmax(model, sim, run_fn, c=5.0):
    """Max |x| theo tung kenh dau vao cua moi QLinear (bo hang ngoai lai), va max cap-kenh cua q/k (truoc RoPE).
    run_fn(): ham chay cac clip o che do 'off' (FP)."""
    ch = {}
    hooks = []
    keep_of = lambda r: (r <= c * r.median()) if float(r.median()) > 0 else torch.ones_like(r, dtype=torch.bool)

    def upd(name, v):
        ch[name] = torch.maximum(ch[name], v) if name in ch else v

    def valid(mod, x):
        n = sim.QS.valid.get(mod.valid_key) if mod.valid_key else None
        return x[:, :n] if n else x

    def pre(mod, args):
        x = valid(mod, args[0].detach()).float()
        x2 = x.reshape(-1, x.shape[-1]).abs()
        r = x2.amax(1)
        upd(mod.name, x2[keep_of(r)].amax(0).cpu())

    def post_qk(kind, nh, d):
        def f(mod, args, out):
            y = valid(mod, out.detach()).float().reshape(-1, nh, d)
            r = y.abs().amax(dim=(1, 2))
            y = y[keep_of(r)]
            mag = torch.sqrt(y[..., : d // 2] ** 2 + y[..., d // 2:] ** 2).amax(0)        # [nh, d/2]
            upd(mod.name + ".pair", mag.cpu())
        return f

    cfg = model.language_model.config
    d = cfg.hidden_size // cfg.num_attention_heads
    for m in model.modules():
        if isinstance(m, sim.QLinear):
            hooks.append(m.register_forward_pre_hook(pre))
            if m.name.startswith("llm.") and m.name.endswith(".q_proj"):
                hooks.append(m.register_forward_hook(post_qk("q", cfg.num_attention_heads, d)))
            if m.name.startswith("llm.") and m.name.endswith(".k_proj"):
                hooks.append(m.register_forward_hook(post_qk("k", cfg.num_key_value_heads, d)))
    sim.QS.mode = "off"
    try:
        run_fn()
    finally:
        for h in hooks:
            h.remove()
    return ch


@torch.no_grad()
def apply(model, sim, CH, a_down=None, a_gu=None, a_qkv=None, a_qk=None, a_enc=None):
    """Ap dung lam muot. Moi tham so a_* la alpha (None = tat). Tra ve dict thong ke so s."""
    info = {}
    cfg = model.language_model.config
    Hq, Hkv = cfg.num_attention_heads, cfg.num_key_value_heads
    d = cfg.hidden_size // Hq
    rep = Hq // Hkv
    for j, L in enumerate(model.language_model.model.layers):
        a, m = L.self_attn, L.mlp
        if a_down is not None:
            s = _smooth(CH[f"llm.{j}.down_proj"], m.down_proj.lin.weight.abs().amax(0), a_down)
            _mul_cols(m.down_proj, s)
            inv = (1.0 / s).to(m.down_proj.lin.weight.device)
            _HOOKS.append(m.down_proj.register_forward_pre_hook(
                lambda mod, args, inv=inv: (args[0] * inv.to(args[0].dtype),) + tuple(args[1:])))
            INV[f"llm.{j}.down_proj"] = inv.detach().float().cpu().clone()
            info[f"llm.{j}.down"] = (float(s.min()), float(s.max()))
        if a_gu is not None:
            Wc = torch.maximum(m.gate_proj.lin.weight.abs().amax(0), m.up_proj.lin.weight.abs().amax(0))
            s = _smooth(CH[f"llm.{j}.gate_proj"], Wc, a_gu)
            _div_vec(L.post_attention_layernorm.weight, s)
            _mul_cols(m.gate_proj, s); _mul_cols(m.up_proj, s)
            info[f"llm.{j}.gu"] = (float(s.min()), float(s.max()))
        if a_qkv is not None:
            Wc = torch.maximum(torch.maximum(a.q_proj.lin.weight.abs().amax(0), a.k_proj.lin.weight.abs().amax(0)),
                               a.v_proj.lin.weight.abs().amax(0))
            s = _smooth(CH[f"llm.{j}.q_proj"], Wc, a_qkv)
            _div_vec(L.input_layernorm.weight, s)
            for p in (a.q_proj, a.k_proj, a.v_proj):
                _mul_cols(p, s)
            info[f"llm.{j}.qkv"] = (float(s.min()), float(s.max()))
        if a_qk is not None:
            qm = CH[f"llm.{j}.q_proj.pair"].reshape(Hkv, rep, d // 2).amax(1)           # [Hkv, d/2]
            km = CH[f"llm.{j}.k_proj.pair"]                                              # [Hkv, d/2]
            t = (km.clamp_min(1e-5) / qm.clamp_min(1e-5)).pow(0.5 * a_qk).clamp(1e-2, 1e2)
            t = torch.cat([t, t], dim=1)                                                 # [Hkv, d]
            tq = t.repeat_interleave(rep, 0).reshape(-1)
            _scale_rows(a.q_proj, tq, inverse=False)
            _scale_rows(a.k_proj, t.reshape(-1), inverse=True)
            info[f"llm.{j}.qk"] = (float(t.min()), float(t.max()))
    if a_enc is not None:
        for i, L in enumerate(model.audio_tower.layers):
            a = L.self_attn
            Wc = torch.maximum(torch.maximum(a.q_proj.lin.weight.abs().amax(0), a.k_proj.lin.weight.abs().amax(0)),
                               a.v_proj.lin.weight.abs().amax(0))
            s = _smooth(CH[f"enc.{i}.q_proj"], Wc, a_enc)
            _div_vec(L.self_attn_layer_norm.weight, s); _div_vec(L.self_attn_layer_norm.bias, s)
            for p in (a.q_proj, a.k_proj, a.v_proj):
                _mul_cols(p, s)
            s = _smooth(CH[f"enc.{i}.fc1"], L.fc1.lin.weight.abs().amax(0), a_enc)
            _div_vec(L.final_layer_norm.weight, s); _div_vec(L.final_layer_norm.bias, s)
            _mul_cols(L.fc1, s)
            s = _smooth(CH[f"enc.{i}.out_proj"], a.out_proj.lin.weight.abs().amax(0), a_enc)
            _scale_rows(a.v_proj, s, inverse=True)
            _mul_cols(a.out_proj, s)
            info[f"enc.{i}"] = (float(s.min()), float(s.max()))
    for m in list(_TOUCH.values()):
        _refresh(m)
    return info

Writing qsmooth.py


In [10]:
# ===== 11. Lam muot + hieu chuan scale tren cac clip hoi thoai (giong v10: c=2, scale theo kenh) =====
import qsmooth as qs
STATS = None
def collect(c):
    sim.QS.fp_layers = set()
    sim.QS.mode = "calib"; sim.QS.row_fb = True; sim.QS.row_c = c; sim.QS.out_pc = True
    out = []
    for r in REF:
        sim.set_valid(r["inp"]["feature_attention_mask"]); sim.start_calib()
        sim.forward_logits(model, r["inp"], r["gen"])
        out.append(sim.stats_of_clip())
    sim.QS.mode = "off"; sim.QS.row_fb = False; sim.QS.out_pc = False
    return out
def _run_off():
    for r in REF:
        sim.set_valid(r["inp"]["feature_attention_mask"]); sim.forward_logits(model, r["inp"], r["gen"])
CH = qs.collect_chmax(model, sim, _run_off, c=2.0)
SM_KW = dict(a_down=0.5, a_gu=0.7, a_qkv=0.7, a_qk=1.0, a_enc=0.5)
qs.restore(model); SM_INFO = qs.apply(model, sim, CH, **SM_KW)
sim.QS.mode = "off"; sim.set_valid(REF[0]["inp"]["feature_attention_mask"])
lq0, _, _ = sim.forward_logits(model, REF[0]["inp"], REF[0]["gen"])
d = float((lq0.cpu() - REF[0]["lf"]).abs().max()); print("FP sau lam muot lech", d, "so voi FP goc (phai rat nho)"); assert d < 0.05
t0 = time.time(); STATS = collect(2.0); STATS_C = {2.0: STATS}; print(f"hieu chuan c=2 tren {len(STATS)} clip [{time.time()-t0:.0f}s]")
PC_MARGIN, ROW_C = 1.5, 2.0
def set_int8(scales):
    sim.QS.scales, sim.QS.fp_layers = scales, set()
    sim.QS.score_mode, sim.QS.lm_out, sim.QS.fb_w = "int32", "int32", "int8"
    sim.QS.int32_ops, sim.QS.attn_fp, sim.QS.out_pc, sim.QS.sat = (), (), True, "clip"
def unset_int8():
    sim.QS.mode, sim.QS.row_fb, sim.QS.out_pc, sim.QS.sat = "off", False, False, "edge"

FP sau lam muot lech 6.103515625e-05 so voi FP goc (phai rat nho)
hieu chuan c=2 tren 50 clip [84s]


In [12]:
# ===== 6. Chot scale P1.5, so sanh voi scale cua lan chay v12 (neu co), do anh huong cua lam tron embedding =====
SCALES_ALL = sim.build_scales(STATS, 100.0, 100.0, pc_margin=PC_MARGIN)
set_int8(SCALES_ALL); sim.QS.mode = "off"
print("so khoa scale:", len(SCALES_ALL), "| so clip hieu chuan:", len(STATS))
if SAVED_SCALES:
    try:
        old = json.load(open(SAVED_SCALES))["scales"]
        diffs = []
        for k, v in SCALES_ALL.items():
            if k not in old: continue
            a = v.double().cpu() if torch.is_tensor(v) else torch.tensor([float(v)], dtype=torch.float64)
            b = torch.tensor(old[k], dtype=torch.float64).reshape(-1)
            diffs.append(float(((a - b).abs() / b.abs().clamp_min(1e-30)).max()))
        diffs = np.array(diffs)
        print("so voi %s: %d khoa chung | lech tuong doi lon nhat %.2e, trung vi %.2e, so khoa lech > 1e-3: %d" % (
            os.path.basename(SAVED_SCALES), len(diffs), diffs.max(), np.median(diffs), int((diffs > 1e-3).sum())))
        print("(lech nho = cung scale; lech vai % tro len nghia la clip hieu chuan khac lan chay v12, vd. wav sinh lai)")
    except Exception as e:
        print("Khong doc duoc scale cu:", type(e).__name__, str(e)[:100])

emb = model.language_model.get_input_embeddings()
if "W0" not in globals(): W0 = emb.weight.data.clone()      # luon giu bang embedding GOC; chay lai cell nay van dung
emb.weight.data.copy_(W0)
def int8_logits(ixs):
    out = {}
    set_int8(SCALES_ALL); sim.QS.mode, sim.QS.row_fb = "quant", True
    try:
        for i in ixs:
            r = REF[i]; sim.set_valid(r["inp"]["feature_attention_mask"])
            with torch.no_grad():
                lq, _, _ = sim.forward_logits(model, r["inp"], r["gen"])
            out[i] = lq.cpu()
    finally:
        unset_int8()
    return out
def _cmp(a, b, i):                                          # tat ca tren CPU (full phai o CPU de danh chi muc logit CPU)
    return sim.compare_logits(a, b, REF[i]["full"].cpu(), REF[i]["n0"])
CK = list(range(0, len(REF), max(1, len(REF) // CHECK_CLIPS)))[:CHECK_CLIPS]
LA = int8_logits(CK)
emb.weight.data.copy_(W0.half().float())
LB = int8_logits(CK)
def _top1(L):
    ms = [_cmp(REF[i]["lf"], L[i], i) for i in CK]
    return sum(m["top1"] * m["n"] for m in ms) / sum(m["n"] for m in ms)
agree = sum(_cmp(LA[i], LB[i], i)["top1"] * LA[i].shape[0] for i in CK) / sum(LA[i].shape[0] for i in CK)
print("Lam tron embedding sang fp16 (%d clip): top-1 so voi FP32  fp32-embed %.2f%% -> fp16-embed %.2f%% | INT8(fp32-embed) vs INT8(fp16-embed) trung nhau %.2f%%" % (
    len(CK), 100 * _top1(LA), 100 * _top1(LB), 100 * agree))
if EMBED_DTYPE == "fp16":
    print("EMBED_DTYPE = fp16: GIU bang embedding da lam tron trong mo hinh (de mo phong == goi xuat).")
else:
    emb.weight.data.copy_(W0); print("EMBED_DTYPE = fp32: khoi phuc bang embedding goc.")
del LA, LB

so khoa scale: 1196 | so clip hieu chuan: 50
so voi scales_P15.json: 1196 khoa chung | lech tuong doi lon nhat 0.00e+00, trung vi 0.00e+00, so khoa lech > 1e-3: 0
(lech nho = cung scale; lech vai % tro len nghia la clip hieu chuan khac lan chay v12, vd. wav sinh lai)
Lam tron embedding sang fp16 (20 clip): top-1 so voi FP32  fp32-embed 90.99% -> fp16-embed 90.99% | INT8(fp32-embed) vs INT8(fp16-embed) trung nhau 100.00%
EMBED_DTYPE = fp16: GIU bang embedding da lam tron trong mo hinh (de mo phong == goi xuat).


In [13]:
# ===== 7. Xuat goi: trong so INT8 [K,N], qparams [N,4], tham so FP, scale, manifest =====
import hashlib, pathlib, shutil, sys, datetime
assert sys.byteorder == "little"
set_int8(SCALES_ALL); sim.QS.mode = "off"
S = sim.QS.scales
PKG = pathlib.Path(PKG_DIR)
if PKG.exists():
    print("Xoa goi cu (se tao lai):", PKG); shutil.rmtree(PKG)
PKG.mkdir(parents=True)

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""): h.update(b)
    return h.hexdigest()
class BinW:
    def __init__(s, name, align=64): s.name, s.f, s.n, s.align = name, open(PKG / name, "wb"), 0, align
    def add(s, arr):
        a = np.ascontiguousarray(arr)
        pad = (-s.n) % s.align
        if pad: s.f.write(b"\0" * pad); s.n += pad
        off = s.n; s.f.write(memoryview(a).cast("B")); s.n += a.nbytes
        return dict(file=s.name, offset=off, dtype=str(a.dtype), shape=list(a.shape))
    def close(s): s.f.close()
WB, QB, TB, EB = BinW("weights_int8.bin"), BinW("qparams.bin"), BinW("tensors.bin"), BinW("embed.bin")

GM = {m.name: m for m in model.modules() if isinstance(m, sim.QGemm)}
GPATH = {m.name: n for n, m in model.named_modules() if isinstance(m, sim.QGemm)}
n_enc, n_llm = len(model.audio_tower.layers), len(model.language_model.model.layers)
PLAN = [("conv1", ["conv1"], "audio_encoder_conv", None), ("conv2", ["conv2"], "audio_encoder_conv", None)]
for i in range(n_enc):
    lk = f"enc.{i}"
    PLAN += [(f"{lk}.qkv", [f"{lk}.q_proj", f"{lk}.k_proj", f"{lk}.v_proj"], "audio_encoder_projections", i),
             (f"{lk}.out_proj", [f"{lk}.out_proj"], "audio_encoder_projections", i),
             (f"{lk}.fc1", [f"{lk}.fc1"], "audio_encoder_ffn", i), (f"{lk}.fc2", [f"{lk}.fc2"], "audio_encoder_ffn", i)]
PLAN.append(("proj", ["proj"], "projector", None))
for j in range(n_llm):
    lk = f"llm.{j}"
    PLAN += [(f"{lk}.qkv", [f"{lk}.q_proj", f"{lk}.k_proj", f"{lk}.v_proj"], "llm_attention_projections", j),
             (f"{lk}.o_proj", [f"{lk}.o_proj"], "llm_attention_projections", j),
             (f"{lk}.gate_up", [f"{lk}.gate_proj", f"{lk}.up_proj"], "llm_mlp", j),
             (f"{lk}.down_proj", [f"{lk}.down_proj"], "llm_mlp", j)]
PLAN.append(("lm_head", ["lm_head"], "llm_lm_head", None))
assert sorted(n for _, ms, _, _ in PLAN for n in ms) == sorted(GM), "PLAN khong khop cac GEMM da boc"

GEMMS = []
shift_all = []
for gname, members, fn, layer in PLAN:
    ms = [GM[n] for n in members]
    K = ms[0].W8.shape[1]
    assert all(m.W8.shape[1] == K for m in ms)
    sin = S[ms[0].name + ".in"]
    assert all(S[m.name + ".in"] == sin for m in ms), f"{gname}: cac thanh vien khong chung scale vao -> khong gop duoc"
    W8cat = torch.cat([m.W8 for m in ms], 0)                       # [N,K]
    N = W8cat.shape[0]
    int32 = (gname == "lm_head")
    ent = dict(name=gname, function=fn, layer=layer, K=K, N=N, sin=float(sin), out="int32" if int32 else "int8",
               rowfb=bool(ms[0].ROWFB), valid_key=ms[0].valid_key, members=[], bias_present=any(m.bias is not None for m in ms))
    c0 = 0
    for m in ms:
        ent["members"].append(dict(name=m.name, cols=[c0, c0 + m.W8.shape[0]], has_bias=m.bias is not None)); c0 += m.W8.shape[0]
    ent["B"] = WB.add(W8cat.t().contiguous().cpu().numpy())        # [K,N] row-major
    sw = torch.cat([m.sw for m in ms]).cpu()
    ent["sw"] = TB.add(sw.float().numpy())
    if int32: ent["sw_f64"] = TB.add(sw.double().numpy())          # lm_head: y = float32(acc * (sin * sw)) tinh bang double
    ent["bias_f32"] = TB.add(torch.cat([(m.bias.float() if m.bias is not None else torch.zeros(m.W8.shape[0], device=m.W8.device)) for m in ms]).cpu().numpy())
    if not int32:
        qp_parts, so_parts, bi_nonzero = [], [], 0
        for m in ms:
            sout = m.sout_vec()
            mult, shift, bi = m._qp(sin, sout)
            bi = torch.zeros_like(mult) if bi is None else bi
            assert int(mult.max()) <= 2 ** 31 - 1 and int(mult.min()) >= 1 and int(shift.min()) >= 0 and int(shift.max()) <= 62
            assert int(bi.min()) >= -2 ** 31 and int(bi.max()) <= 2 ** 31 - 1
            qp_parts.append(torch.stack([bi, mult, shift, torch.zeros_like(shift)], 1).to(torch.int32).cpu())
            so_parts.append(sout.float().cpu())
            shift_all.append((int(shift.min()), int(shift.max())))
        ent["qparams"] = QB.add(torch.cat(qp_parts, 0).numpy())    # int32 [N,4] = bias, mult, shift, zp
        ent["sout"] = TB.add(torch.cat(so_parts).numpy())          # float32 [N]: y = v * sout
    GEMMS.append(ent)
print("GEMM da xuat:", len(GEMMS), "| weights_int8.bin %.1f MB" % (WB.n / 2**20), "| shift min/max: %d / %d" % (min(a for a, _ in shift_all), max(b for _, b in shift_all)))

# --- tham so FP con lai: moi muc state_dict KHONG thuoc GEMM nao (de bao dam khong sot)
sd = model.state_dict()
covered = set()
for nm, p in GPATH.items():
    for suf in ("lin.weight", "lin.bias", "conv.weight", "conv.bias"): covered.add(f"{p}.{suf}")
TENS, EMBT = {}, {}
for k in sorted(sd):
    if k in covered: continue
    t = sd[k].detach().cpu()
    if k.endswith("embed_tokens.weight"):
        EMBT[k] = EB.add(t.half().numpy() if EMBED_DTYPE == "fp16" else t.float().numpy())
    elif t.is_floating_point():
        TENS[k] = TB.add(t.float().numpy())
    else:
        print("  canh bao: tensor so nguyen", k, "luu nguyen dang", t.dtype); TENS[k] = TB.add(t.numpy())
for k, v in qs.INV.items(): TENS[k.replace("down_proj", "down_inv")] = TB.add(v.float().numpy())
assert len(EMBT) == 1, f"can dung 1 bang embedding, thay {list(EMBT)}"
# mel
fe = processor.feature_extractor
try:
    mel_filters = np.asarray(fe.mel_filters, dtype=np.float32)
    TENS["mel_filters"] = TB.add(mel_filters)
    HANN = (0.5 - 0.5 * np.cos(2 * np.pi * np.arange(fe.n_fft) / fe.n_fft)).astype(np.float32)
    TENS["hann_window"] = TB.add(HANN)
except Exception as e:
    mel_filters = HANN = None; print("canh bao: khong lay duoc bo loc mel:", type(e).__name__, str(e)[:80])
for w in (WB, QB, TB, EB): w.close()
print("tensors.bin %.1f MB (%d tensor) | embed.bin %.1f MB (%s) | qparams.bin %.1f MB" % (TB.n / 2**20, len(TENS), EB.n / 2**20, EMBED_DTYPE, QB.n / 2**20))
print("state_dict: %d muc | GEMM %d | FP %d | embedding %d | CHUA PHU: %s" % (len(sd), len([k for k in sd if k in covered]), len(TENS) - len(qs.INV) - (2 if mel_filters is not None else 0), len(EMBT),
      [k for k in sd if k not in covered and k not in TENS and k not in EMBT]))

GEMM da xuat: 228 | weights_int8.bin 1080.8 MB | shift min/max: 35 / 48
tensors.bin 18.8 MB (206 tensor) | embed.bin 266.7 MB (fp16) | qparams.bin 10.3 MB
state_dict: 780 muc | GEMM 599 | FP 180 | embedding 1 | CHUA PHU: []


In [14]:
# ===== 8. Scale attention, kich thuoc, tokenizer, manifest =====
ac, tc = model.config.audio_config, model.config.text_config
L0e, L0l = model.audio_tower.layers[0], model.language_model.model.layers[0]
hd = tc.hidden_size // tc.num_attention_heads
rope_theta = getattr(tc, "rope_theta", None) or (getattr(tc, "rope_parameters", None) or {}).get("rope_theta")
DIMS = dict(
    enc=dict(layers=n_enc, d_model=ac.d_model, heads=ac.encoder_attention_heads, ffn=ac.encoder_ffn_dim, mel=ac.num_mel_bins,
             max_pos=ac.max_source_positions, ln_eps=float(L0e.self_attn_layer_norm.eps), q_scaling=float(L0e.self_attn.scaling)),
    llm=dict(layers=n_llm, hidden=tc.hidden_size, heads=tc.num_attention_heads, kv_heads=tc.num_key_value_heads, head_dim=hd,
             inter=tc.intermediate_size, vocab=tc.vocab_size, rope_theta=rope_theta, rms_eps=float(L0l.input_layernorm.variance_epsilon),
             tie_word_embeddings=bool(tc.tie_word_embeddings)),
    audio=dict(sampling_rate=int(SR), n_fft=int(fe.n_fft), hop=int(fe.hop_length), n_samples=int(fe.n_samples), feature_size=int(fe.feature_size)))
ATTN = {}
for j in range(n_llm):
    lk = f"llm.{j}"; sq, sk = S[lk + ".attn.q"], S[lk + ".attn.k"]
    ATTN[lk] = dict(kind="llm", sq=float(sq), sk=float(sk), scaling=hd ** -0.5, fac=float(sq * sk * hd ** -0.5), v_scale="sout slice cols of v_proj in group %s.qkv" % lk)
for i in range(n_enc):
    lk = f"enc.{i}"; a_ = model.audio_tower.layers[i].self_attn
    sq, sk = S[lk + ".q_proj.out"] * a_.scaling, S[lk + ".k_proj.out"]
    ATTN[lk] = dict(kind="enc", sq=float(sq), sk=float(sk), scaling=1.0, fac=float(sq * sk), v_scale="sout slice cols of v_proj in group %s.qkv" % lk)
SCALARS = {k: float(v) for k, v in S.items() if not torch.is_tensor(v)}
json.dump(SCALARS, open(PKG / "scales.json", "w"), indent=0)
processor.save_pretrained(str(PKG / "hf_processor"))
os.makedirs(PKG / "reference", exist_ok=True)
for f in ("qint8_sim.py", "qsmooth.py"): shutil.copy(f, PKG / "reference" / f)
CLIPS = [dict(wav=os.path.basename(p), sha256=sha256_file(p), question=q) for p, q in zip(WAVS, QUESTIONS)]
MANIFEST = dict(
    format="qwen2audio-int8-pkg/1", created=datetime.datetime.now().isoformat(timespec="seconds"),
    model=dict(id=MODEL_ID, revision=REV, transformers=transformers.__version__, torch=torch.__version__),
    quant_config=dict(name="P1.5", row_c=ROW_C, pc_margin=PC_MARGIN, out_pc=True, sat="clip", score_mode="int32", lm_out="int32",
                      int32_ops=[], smooth=SM_KW, act_scale="static symmetric per-tensor (zp=0)", weight_scale="per-output-channel",
                      embed_dtype=EMBED_DTYPE),
    dims=DIMS, decode=dict(greedy=True, repetition_penalty=1.1, eos_id=int(IM_END_ID), audio_token_id=int(AUDIO_ID), max_new=MAX_NEW,
                          prompt=PROMPT, note="so <|AUDIO|> = proj_valid_rows; 30 s audio -> input_features [128,3000] -> enc 1500 -> proj 750"),
    embedding=dict(tensor=list(EMBT.keys())[0], **list(EMBT.values())[0]),
    gemms=GEMMS, tensors=TENS, attention=ATTN,
    calibration=dict(n_clips=len(STATS), clips=CLIPS, note="scale = max qua cac clip (per-tensor kich hoat; per-kenh dau ra x pc_margin)"),
    reference_results=dict(source="quiz_report_{kaggle,colab}.json cua v12 (mo phong, khong bit-true RTL)",
                           teacher_forced_top1_vs_fp32=0.9196, tokens=3410, quiz_s1_correct=dict(fp32=111, bf16=107, int8=107, n=150)),
    files={})
(PKG / "MANIFEST.json").write_text(json.dumps(MANIFEST, indent=1))
print("MANIFEST.json:", len(GEMMS), "GEMM,", len(TENS), "tensor FP,", len(ATTN), "khoi attention")

MANIFEST.json: 228 GEMM, 206 tensor FP, 56 khoi attention


In [15]:
# ===== 9. Vector kiem thu tung GEMM (cung dinh dang goi kv260_all: chay duoc bang run_all_functions.py tren board) =====
# Lay dau vao THAT cua GEMM trong 1 lan forward INT8 (prefill + buoc decode dau tien) cua clip 0; golden tinh bang so hoc so nguyen cua mo phong;
# doi chieu truc tiep voi dau ra ma mo phong thuc su tra ve trong lan forward do (tru hang fallback FP).
VDIR = PKG / "vectors"
DEVX = next(model.parameters()).device
def _layers(n): return sorted({0, n // 2, n - 1})
SEL_E, SEL_L = _layers(n_enc), _layers(n_llm)
sel_groups = [g for g in GEMMS if g["layer"] is None or g["layer"] in (SEL_E if g["name"].startswith("enc.") else SEL_L)]
sel_members = sorted({m["name"] for g in sel_groups for m in g["members"]})
CAP = {}
def _wrap(name):
    m = GM[name]; orig = m.gemm
    def w(x2d, int32_out=False, _o=orig, _n=name, _m=m):
        ph = "decode" if x2d.shape[0] == 1 else "prefill"
        y = _o(x2d, int32_out=int32_out)
        if (_n, ph) not in CAP:
            x = x2d.detach().float(); M = x.shape[0]
            if ph == "prefill" and M > VEC_ROWS:
                even = torch.linspace(0, M - 1, VEC_ROWS).round().long().to(x.device)
                sat = ((x.abs().amax(1) > 127.5 * S[_n + ".in"]).nonzero().flatten()[:8])
                idx = torch.unique(torch.cat([even, sat]))
            else:
                idx = torch.arange(M, device=x.device)
            fbm = getattr(_m, "last_fb", None)
            CAP[(_n, ph)] = dict(Mfull=int(M), idx=idx.cpu(), x=x[idx].cpu(), y=y[idx].detach().float().cpu(),
                                 fb=(fbm[idx].cpu() if fbm is not None and fbm.numel() == M else torch.zeros(len(idx), dtype=torch.bool)))
        return y
    m.gemm = w
for n in sel_members: _wrap(n)
r0 = REF[0]; set_int8(SCALES_ALL); sim.reset_fb(); sim.QS.mode, sim.QS.row_fb = "quant", True; sim.set_valid(r0["inp"]["feature_attention_mask"])
try:
    with torch.no_grad():
        model.generate(**r0["inp"], max_new_tokens=2, do_sample=False, repetition_penalty=1.1, eos_token_id=IM_END_ID,
                       pad_token_id=processor.tokenizer.pad_token_id, use_cache=True)
finally:
    unset_int8()
    for n in sel_members:
        if "gemm" in GM[n].__dict__: del GM[n].__dict__["gemm"]
print("da bat dau vao/ra that cua", len(CAP), "(GEMM, pha)")

def _tag(s): return s.replace(".", "_")
VFUNCS, nfiles, bad = {}, 0, 0
set_int8(SCALES_ALL)
for g in sel_groups:
    fn, name = g["function"], g["name"]
    fdir = VDIR / "functions" / fn; gdir = fdir / _tag(name); gdir.mkdir(parents=True, exist_ok=True)
    ms = [GM[mm["name"]] for mm in g["members"]]
    W8cat = torch.cat([m.W8 for m in ms], 0).to(DEVX)
    sin = S[ms[0].name + ".in"]
    Bk = np.fromfile(PKG / g["B"]["file"], dtype=np.int8, count=g["K"] * g["N"], offset=g["B"]["offset"]).reshape(g["K"], g["N"])
    assert np.array_equal(Bk, W8cat.t().cpu().numpy())
    Bk.tofile(gdir / "B.bin")
    if g["out"] == "int8":
        qp = np.fromfile(PKG / g["qparams"]["file"], dtype=np.int32, count=g["N"] * 4, offset=g["qparams"]["offset"]).reshape(g["N"], 4)
    else:
        qp = np.tile(np.array([[0, 1, 0, 0]], dtype=np.int32), (g["N"], 1))          # lm_head: che do INT32, qparams khong dung
    qp.tofile(gdir / "params.bin")
    mult = torch.tensor(qp[:, 1], dtype=torch.int64, device=DEVX); shift = torch.tensor(qp[:, 2], dtype=torch.int64, device=DEVX)
    bi = torch.tensor(qp[:, 0], dtype=torch.int64, device=DEVX)
    cases = []
    for ph in ("prefill", "decode"):
        c0 = CAP.get((g["members"][0]["name"], ph))
        if c0 is None: continue
        xf = c0["x"].to(DEVX)
        x8 = torch.clamp(torch.round(xf / sin), -128, 127)
        acc = sim.int_mm(x8, W8cat)                                                  # float64 [M,N], so nguyen
        assert float(acc.abs().max()) < 2 ** 31
        a32 = acc.long().to(torch.int32).cpu().numpy()
        # doi chieu voi dau ra that cua mo phong (tung thanh vien), bo cac hang fallback FP
        fbmask = torch.zeros(len(c0["idx"]), dtype=torch.bool)
        for mm in g["members"]: fbmask |= CAP[(mm["name"], ph)]["fb"]
        ok = ~fbmask
        if g["out"] == "int8":
            c8 = sim.requant(acc, mult, shift, bi).to(torch.int8).cpu().numpy()
            sout = torch.tensor(np.fromfile(PKG / g["sout"]["file"], dtype=np.float32, count=g["N"], offset=g["sout"]["offset"]))
            ref = torch.cat([CAP[(mm["name"], ph)]["y"] for mm in g["members"]], 1)
            mine = torch.from_numpy(c8).float() * sout[None, :]
        else:
            c8 = None
            sw64 = torch.tensor(np.fromfile(PKG / g["sw_f64"]["file"], dtype=np.float64, count=g["N"], offset=g["sw_f64"]["offset"]))
            ref = CAP[("lm_head", ph)]["y"]
            mine = (torch.from_numpy(a32).double() * (sin * sw64)[None, :]).float()
        if ok.any():
            same = bool((ref[ok] == mine[ok]).all())
            if not same: bad += 1; print("  KHONG KHOP voi dau ra mo phong:", name, ph, "so o lech:", int((ref[ok] != mine[ok]).sum()))
        tag = ph
        xf8 = x8.to(torch.int8).cpu().numpy()
        xf8.tofile(gdir / f"A_{tag}.bin"); a32.tofile(gdir / f"C32_{tag}.bin")
        files = {"A": f"{_tag(name)}/A_{tag}.bin", "B": f"{_tag(name)}/B.bin", "params": f"{_tag(name)}/params.bin", "golden32": f"{_tag(name)}/C32_{tag}.bin"}
        if c8 is not None: c8.tofile(gdir / f"C8_{tag}.bin"); files["golden8"] = f"{_tag(name)}/C8_{tag}.bin"
        cid = name if g["layer"] is None else "L%02d/%s" % (g["layer"], name.split(".", 2)[-1])
        cid = cid + ("@M1" if ph == "decode" else "")
        cases.append(dict(id=cid, name=name, layer=g["layer"], phase=ph, M=int(xf8.shape[0]), K=g["K"], N=g["N"], mode=g["out"],
                          rows=c0["idx"].tolist(), m_full=c0["Mfull"],
                          calls_per_forward=(n_enc if name.startswith("enc.") else n_llm if name.startswith("llm.") and g["layer"] is not None else 1),
                          n_fallback_rows=int(fbmask.sum()),
                          files={k: dict(path=v, sha256=sha256_file(fdir / v)) for k, v in files.items()}))
        nfiles += len(files)
    VFUNCS.setdefault(fn, []).extend(cases)
for fn, cs in VFUNCS.items():
    json.dump(dict(function=fn, cases=cs), open(VDIR / "functions" / fn / "function.json", "w"), indent=1)
json.dump(dict(format="qwen2audio-int8-vectors/1", functions={fn: dict(dir=f"functions/{fn}", cases=len(cs)) for fn, cs in VFUNCS.items()},
               note="cung dinh dang goi kv260_all: sudo python3 run_all_functions.py --pkg <thu muc nay> --repeat 3"),
          open(VDIR / "INDEX.json", "w"), indent=1)
print("vector: %d function, %d case, %d file | doi chieu voi dau ra mo phong: %s" % (len(VFUNCS), sum(len(c) for c in VFUNCS.values()), nfiles, "TAT CA KHOP" if bad == 0 else f"{bad} case LECH"))
assert bad == 0, "vector khong khop dau ra mo phong -> loi trong luong xuat, dung lai"

da bat dau vao/ra that cua 65 (GEMM, pha)
vector: 7 function, 41 case, 203 file | doi chieu voi dau ra mo phong: TAT CA KHOP


In [16]:
# ===== 10. Vector dau-cuoi: token + top-5 logit (FP32 va INT8 mo phong) cho vai clip =====
GDIR = PKG / "golden"; GDIR.mkdir(exist_ok=True)
GC = sorted({int(round(x)) for x in np.linspace(0, len(REF) - 1, min(GOLDEN_CLIPS, len(REF)))})
GOLD = []
for i in GC:
    r = REF[i]; inp = r["inp"]; sim.set_valid(inp["feature_attention_mask"])
    set_int8(SCALES_ALL); sim.reset_fb(); sim.QS.mode, sim.QS.row_fb = "quant", True
    try:
        with torch.no_grad():
            g = model.generate(**inp, max_new_tokens=MAX_NEW, do_sample=False, repetition_penalty=1.1, eos_token_id=IM_END_ID,
                               pad_token_id=processor.tokenizer.pad_token_id, use_cache=True)
            sim.reset_fb()
            lq, _, _ = sim.forward_logits(model, inp, r["gen"])
    finally:
        unset_int8()
    gen8 = g[0, inp["input_ids"].shape[1]:].cpu()
    text8 = processor.batch_decode(gen8[None], skip_special_tokens=True)[0].strip()
    d = GDIR / f"clip_{i:02d}"; d.mkdir(exist_ok=True)
    inp["input_features"].detach().float().cpu().numpy().tofile(d / "input_features.f32")
    inp["input_ids"].cpu().numpy().astype(np.int32).tofile(d / "input_ids.i32")
    inp["feature_attention_mask"].cpu().numpy().astype(np.int32).tofile(d / "feature_attention_mask.i32")
    r["gen"].cpu().numpy().astype(np.int32).tofile(d / "fp32_gen_ids.i32")
    gen8.numpy().astype(np.int32).tofile(d / "int8_gen_ids.i32")
    tf = {}
    for tag, L in (("fp32", r["lf"]), ("int8", lq.cpu())):
        v, ix = torch.topk(L, 5, dim=-1); v.numpy().astype(np.float32).tofile(d / f"tf_{tag}_top5_vals.f32"); ix.numpy().astype(np.int32).tofile(d / f"tf_{tag}_top5_ids.i32")
    same = int((gen8.numel() == r["gen"].numel()) and bool((gen8 == r["gen"].cpu()).all()))
    fd = next((t for t in range(min(len(gen8), len(r["gen"]))) if int(gen8[t]) != int(r["gen"][t])), min(len(gen8), len(r["gen"])))
    meta = dict(clip=i, wav=os.path.basename(WAVS[i]), question=QUESTIONS[i], fp32_text=r["text"], int8_text=text8,
                shapes=dict(input_features=list(inp["input_features"].shape), input_ids=list(inp["input_ids"].shape),
                            feature_attention_mask=list(inp["feature_attention_mask"].shape), fp32_gen=int(r["gen"].numel()), int8_gen=int(gen8.numel()),
                            n_logit_rows=int(r["lf"].shape[0]), vocab=int(r["lf"].shape[1])),
                int8_equals_fp32=bool(same), first_divergence=fd, valid=sim.QS.valid)
    json.dump(meta, open(d / "meta.json", "w"), indent=1, ensure_ascii=False); GOLD.append(meta)
    print(f"clip {i:2d}: FP32 {meta['shapes']['fp32_gen']} token | INT8 {meta['shapes']['int8_gen']} token | giong het: {bool(same)} | lech dau tai token {fd}")
    print("    INT8:", text8[:200])

clip  0: FP32 91 token | INT8 92 token | giong het: False | lech dau tai token 19
    INT8: That's an interesting question! Personally, I think the universe would be incredibly peaceful if it were vast enough to accommodate all life forms without any signs of conflict or destruction. On the 
clip 12: FP32 24 token | INT8 14 token | giong het: False | lech dau tai token 3
    INT8: The ancient name of Hnepu is Haohn.
clip 24: FP32 78 token | INT8 77 token | giong het: False | lech dau tai token 21
    INT8: That's an interesting question! If I could switch jobs with anyone for a single day, I would try the role of Customer Service Representative. It requires minimal interaction with customers, which can 
clip 37: FP32 9 token | INT8 9 token | giong het: True | lech dau tai token 9
    INT8: The chemical symbol for gold is Au.
clip 49: FP32 26 token | INT8 30 token | giong het: False | lech dau tai token 11
    INT8: The heart is responsible for pumping blood. It pumps blood throughout t

In [17]:
# ===== 12. Kiem tra lai tu file, kiem tra mel, ghi FORMAT.md + SHA256SUMS =====
# (a) doc lai MOI GEMM tu file nhi phan, doi chieu chinh xac voi mo hinh trong bo nho
set_int8(SCALES_ALL)                 # sout_vec() can out_pc = True
nbad = 0
for g in GEMMS:
    ms = [GM[m["name"]] for m in g["members"]]
    Bk = np.fromfile(PKG / g["B"]["file"], dtype=np.int8, count=g["K"] * g["N"], offset=g["B"]["offset"]).reshape(g["K"], g["N"])
    nbad += int(not np.array_equal(Bk, torch.cat([m.W8 for m in ms], 0).t().cpu().numpy()))
    sw = np.fromfile(PKG / g["sw"]["file"], dtype=np.float32, count=g["N"], offset=g["sw"]["offset"])
    nbad += int(not np.array_equal(sw, torch.cat([m.sw for m in ms]).float().cpu().numpy()))
    if g["out"] == "int8":
        qp = np.fromfile(PKG / g["qparams"]["file"], dtype=np.int32, count=g["N"] * 4, offset=g["qparams"]["offset"]).reshape(g["N"], 4)
        parts = []
        for m in ms:
            mult, shift, bi = m._qp(S[ms[0].name + ".in"], m.sout_vec()); bi = torch.zeros_like(mult) if bi is None else bi
            parts.append(torch.stack([bi, mult, shift, torch.zeros_like(shift)], 1).to(torch.int32).cpu())
        nbad += int(not np.array_equal(qp, torch.cat(parts, 0).numpy()))
unset_int8()
print("(a) doc lai %d GEMM tu file: %s" % (len(GEMMS), "KHOP HOAN TOAN" if nbad == 0 else f"{nbad} LECH"))
assert nbad == 0
for k, ent in list(TENS.items())[:]:
    arr = np.fromfile(PKG / ent["file"], dtype=np.dtype(ent["dtype"]), count=int(np.prod(ent["shape"])), offset=ent["offset"]).reshape(ent["shape"])
    if k in sd: assert np.array_equal(arr, sd[k].detach().cpu().float().numpy()), k
print("(a) %d tensor FP doc lai khop state_dict" % len(TENS))

# (b) mel: cai dat numpy theo dung mo ta trong FORMAT.md so voi bo xu ly cua HF (chi canh bao neu lech)
MEL_ERR = None
try:
    import librosa
    a = librosa.load(WAVS[GC[0]], sr=SR)[0].astype(np.float32)
    a = np.pad(a, (0, max(0, DIMS["audio"]["n_samples"] - len(a))))[:DIMS["audio"]["n_samples"]]
    n_fft, hop = DIMS["audio"]["n_fft"], DIMS["audio"]["hop"]
    ap = np.pad(a, (n_fft // 2, n_fft // 2), mode="reflect")
    nfr = 1 + (len(ap) - n_fft) // hop
    fr = np.stack([ap[t * hop: t * hop + n_fft] * HANN for t in range(nfr)])
    mag = np.abs(np.fft.rfft(fr.astype(np.float64), axis=1)) ** 2                     # [T, 201]
    mel = mag @ mel_filters.astype(np.float64)                                      # [T, 128]
    lg = np.log10(np.maximum(mel, 1e-10))[:-1].T                                    # bo khung cuoi -> [128, 3000]
    lg = (np.maximum(lg, lg.max() - 8.0) + 4.0) / 4.0
    ref_f = REF[GC[0]]["inp"]["input_features"][0].detach().cpu().numpy()
    nv = int(REF[GC[0]]["inp"]["feature_attention_mask"].sum())
    MEL_ERR = float(np.abs(lg[:, :nv] - ref_f[:, :nv]).max())
    print("(b) log-mel numpy so voi HF tren %d khung hop le: lech lon nhat %.2e %s" % (nv, MEL_ERR, "(OK)" if MEL_ERR < 1e-3 else "(LECH LON: mo ta mel trong FORMAT.md can doi chieu lai)"))
except Exception as e:
    print("(b) bo qua kiem tra mel:", type(e).__name__, str(e)[:100])

# (c) FORMAT.md
e_, l_ = DIMS["enc"], DIMS["llm"]
FORMAT = f"""# Goi INT8 Qwen2-Audio (`qwen2audio-int8-pkg/1`)

Sinh boi notebook export. **Mo hinh tham chieu la `reference/qint8_sim.py` + `reference/qsmooth.py`** (mo phong GPU); goi nay chua du lieu da luong tu
va cac quy uoc de runtime C++/PS tai tao dung phep tinh do. Mo phong **chua bit-true voi RTL**: kiem tra bang `vectors/` tren board.

## Tep
| tep | noi dung |
|---|---|
| `MANIFEST.json` | cau hinh, kich thuoc, danh sach GEMM / tensor (file, offset, dtype, shape), scale attention, clip hieu chuan, sha256 |
| `weights_int8.bin` | trong so INT8 cua tung GEMM, dang **B[K,N] row-major** (= chuyen vi cua [N,K] per-channel), moi khoi can 64 byte |
| `qparams.bin` | `int32 [N,4]` moi GEMM = (bias, mult, shift, zp), dung cho npu_qparam_t; zp = 0 |
| `tensors.bin` | cac tensor FP (float32, rieng `sw_f64` cua lm_head la float64): sw, sout, bias_f32, norm, embed_positions cua encoder, `down_inv`, bo loc mel, cua so hann |
| `embed.bin` | bang embedding ({EMBED_DTYPE}) [vocab, hidden], tra cuu theo token id |
| `scales.json` | moi scale vo huong (khoa `<gemm>.in`, `<gemm>.out`, `llm.<j>.attn.q/.k`, ...) |
| `hf_processor/` | tokenizer + preprocessor cua HF |
| `vectors/` | vector kiem thu tung GEMM (dinh dang `kv260_all`, chay bang `run_all_functions.py --pkg vectors`) |
| `golden/clip_XX/` | vector dau-cuoi: dau vao (mel, input_ids), token FP32 / INT8, top-5 logit teacher-forced |
| `reference/` | `qint8_sim.py`, `qsmooth.py` (dac ta chinh xac) |
| `SHA256SUMS` | `sha256sum -c SHA256SUMS` sau khi copy sang board |

Tat ca little-endian. Moi muc trong `MANIFEST.json` co `file`, `offset` (byte), `dtype`, `shape`.

## GEMM
`C[M,N] = A[M,K] x B[K,N]`. **Gop**: q|k|v thanh `qkv`, gate|up thanh `gate_up` (cot lien tiep, xem `members[].cols`); cac thanh vien dung chung scale vao `sin`.
Encoder: q|k|v cung gop (k_proj khong bias -> bias 0). conv1/conv2 la GEMM voi im2col, thu tu cot `k*C + c` (k = tap, c = kenh), padding 1, stride 1 / 2.
* Luong tu vao (PS, truoc khi gui PL): `x8 = clamp(round_half_even(x / sin), -128, 127)` (chia float32, **khong** nhan nghich dao), `sin` = `gemm.sin` (float64 -> cast float32).
* PL: `acc = sum_k x8*B` (INT32), roi voi moi kenh n: `t = acc + bias[n]; q = (t*mult[n] + (shift ? 1<<(shift-1) : 0)) >> shift; v = clamp(q + zp, -128, 127)` (dich so hoc).
* Giai luong tu (PS): `y = float32(v) * sout[n]` (float32). `sout` theo **kenh dau ra** (`out_pc`).
* `out == "int32"` (chi `lm_head`): lay `acc` tho, `y = float32(double(acc) * (sin * sw_f64[n]))`; khong requant. `lm_head` khong co bias.
* **Fallback hang FP** (chi GEMM co `rowfb`; conv thi khong): hang `i` bi danh dau neu `max|x[i]| > 127.5*sin` HOAC bat ky `|(acc+bias)*sin*sw/sout| > 127.5` o dau ra (`sat="clip"`).
  Hang bi danh dau tinh tren PS: `y = x_fp32 @ (B_int8 * sw[n]) + bias_f32` (trong so INT8 giai luong tu, **khong** can trong so FP goc) va thay vao `y`. `x_fp32` la dau vao truoc luong tu.
  Ty le hang FP do duoc: ~1,4% (teacher-forced), prefill ~0,8%, decode ~3,1% (mo phong).
* `down_proj`: PS nhan dau vao (`silu(gate)*up`) voi `down_inv` (tensor `llm.<j>.down_inv`, float32 theo kenh) **truoc** khi luong tu.

## Hinh dang model ({e_['layers']} lop encoder, {l_['layers']} lop LLM)
Encoder (d={e_['d_model']}, {e_['heads']} head, ffn {e_['ffn']}, LayerNorm eps {e_['ln_eps']}, `q` nhan `scaling = {e_['q_scaling']:.6g}`): conv1+GELU, conv2+GELU, cong `embed_positions`,
moi lop: `x += out_proj(attn(LN1(x)))`, `x += fc2(GELU(fc1(LN2(x))))`; sau cung avg-pool (k=2,s=2), LayerNorm, `proj` (-> {l_['hidden']}). GELU/Softmax/LayerNorm: FP32 nhu HF transformers {transformers.__version__}.
LLM (hidden {l_['hidden']}, {l_['heads']} head, {l_['kv_heads']} kv-head, head_dim {l_['head_dim']}, inter {l_['inter']}, vocab {l_['vocab']}, RMSNorm eps {l_['rms_eps']}, rope_theta {l_['rope_theta']}):
embedding -> thay cac vi tri `<|AUDIO|>` bang dau ra `proj` -> moi lop: `RMSNorm -> qkv -> RoPE(q,k) -> attention (KV cache) -> o_proj -> +`, `RMSNorm -> gate_up -> silu(gate)*up -> *down_inv -> down_proj -> +`; norm cuoi -> `lm_head` (INT32) ->
phat lap lai `repetition_penalty = 1.1` (tren lich su gom prompt + token da sinh: logit > 0 chia 1,1; <= 0 nhan 1,1) -> argmax (greedy).
Attention luong tu (xem `make_attention` trong `reference/qint8_sim.py`): `q8 = clamp(round(q/sq))`, `k8 = clamp(round(k/sk))` (q,k **sau RoPE**), `V8` = dau ra INT8 cua `v_proj` (scale theo kenh = `sout` cot v),
diem `acc*fac` (INT32 -> FP32), softmax FP32 (+mask), `P8 = round(p / (max_hang/127))`, `O = (P8 @ V8) * sp * v_scale`; cac hang Q / cot K,V bao hoa hoac danh dau ngoai lai duoc tinh FP. Tham so: `MANIFEST.attention`.
Ghi chu: day la dac ta tu mo phong; **ma tham chieu moi la nguon chinh xac**.

## Mel
`n_fft={DIMS['audio']['n_fft']}`, hop `{DIMS['audio']['hop']}`, audio pad 0 den `{DIMS['audio']['n_samples']}` mau, reflect-pad `n_fft/2` hai ben, cua so hann (`hann_window`), `|STFT|^2`, nhan `mel_filters` [{DIMS['audio']['n_fft']//2+1},{DIMS['audio']['feature_size']}],
`log10(max(., 1e-10))`, bo khung cuoi, `max(., max-8)`, `(x+4)/4` -> `input_features [128, 3000]`. `feature_attention_mask` = so khung hop le theo do dai audio that.
Sai lech cai dat numpy so voi HF (kiem tra trong notebook): {MEL_ERR if MEL_ERR is not None else 'chua kiem tra'}.

## Kiem tra tren board (khong doi cau hinh board)
```
sha256sum -c SHA256SUMS
sudo python3 run_all_functions.py --pkg vectors --repeat 3 --fclk-mhz 100     # tools/ cua kv260_all; chi chay khi khong ai khac dung board
```
"""
(PKG / "FORMAT.md").write_text(FORMAT, encoding="utf-8")

# (d) sha256 toan bo goi
files = sorted(p for p in PKG.rglob("*") if p.is_file() and p.name not in ("SHA256SUMS",))
sums = {str(p.relative_to(PKG)): sha256_file(p) for p in files if p.name != "MANIFEST.json"}
MANIFEST["files"] = {k: dict(sha256=v, bytes=(PKG / k).stat().st_size) for k, v in sums.items() if "/" not in k}
(PKG / "MANIFEST.json").write_text(json.dumps(MANIFEST, indent=1))
sums["MANIFEST.json"] = sha256_file(PKG / "MANIFEST.json")
(PKG / "SHA256SUMS").write_text("".join(f"{h}  {k}\n" for k, h in sorted(sums.items())))
tot = sum((PKG / k).stat().st_size for k in sums)
print("(d) %d file, tong %.2f GB, SHA256SUMS da ghi" % (len(sums), tot / 2**30))
print("\nGOI XUAT:", PKG)
for k in ("weights_int8.bin", "embed.bin", "tensors.bin", "qparams.bin"): print("  %-18s %8.1f MB" % (k, (PKG / k).stat().st_size / 2**20))
print("  vectors/ %.1f MB | golden/ %.1f MB" % (sum(p.stat().st_size for p in (PKG / "vectors").rglob("*") if p.is_file()) / 2**20, sum(p.stat().st_size for p in (PKG / "golden").rglob("*") if p.is_file()) / 2**20))
print("Copy ca thu muc sang board (scp/rsync tu may cua ban), roi: cd pkg_v1 && sha256sum -c SHA256SUMS")

(a) doc lai 228 GEMM tu file: KHOP HOAN TOAN
(a) 206 tensor FP doc lai khop state_dict
(b) log-mel numpy so voi HF tren 688 khung hop le: lech lon nhat 2.06e-05 (OK)
(d) 253 file, tong 1.65 GB, SHA256SUMS da ghi

GOI XUAT: <PKG_DIR>
  weights_int8.bin     1080.8 MB
  embed.bin             266.7 MB
  tensors.bin            18.8 MB
  qparams.bin            10.3 MB
  vectors/ 292.1 MB | golden/ 7.4 MB
Copy ca thu muc sang board (scp/rsync tu may cua ban), roi: cd pkg_v1 && sha256sum -c SHA256SUMS
